# 🧪 **Proyecto Júpiter: FakeNews - Notebook 1: Ingesta de Fuentes Crudas, Auditoría Estática de Calidad (Data Quality) y Normalización**

> **Punto de partida del pipeline analítico.** Este cuaderno materializa la primera fase del proceso de extracción, transformación y carga (ETL), actuando como la **Capa de Ingesta y Staging (JSON crudos en datos_raw)** del proyecto. Su propósito central es absorber las 8 fuentes de información crudas en formato JSON, ejecutar un motor de auditoría estática automatizado para diagnosticar la salud de cada variable de forma aislada y establecer la línea base de control de calidad antes de proceder a la reestructuración y persistencia relacional.

---

## 📋 **Ficha Técnica de la Fase de Ingesta**

| Campo | Detalle Técnico de la Infraestructura |
|---|---|
| **Autores** | AB, JE, IF & JR - Proyecto Júpiter: Fake News (PontIA) |
| **Naturaleza del proceso** | Ingesta atómica, perfilado estadístico univariable (*Data Profiling*) y tipado de datos. |
| **Origen de los datos** | Capa de datos brutos sin procesar (`datos_raw/` conteniendo los 8 archivos JSON originales). |
| **Volumen controlado** | Procesamiento individualizado de las 8 entidades (*info*, *tipo*, *fake*, *tiempo*, *contenido*, *metadatos*, *pais_idioma*, *compartir_tiempo*). |
| **Mapeo de Control** | Identificación de nulos reales, anomalías de signo, cadenas vacías y el uso del valor centinela `"-1"`. |
| **Cierre del proceso** | Estructuras de datos desanidadas y aplanadas a formato tabular, limpias, tipadas y normalizadas, listas para la carga relacional en MySQL (NB2) y el análisis exploratorio (NB3). |

---

## 🗺️ **Estructura del Pipeline de Control (Los 6 Bloques de Ejecución)**

Para garantizar un diagnóstico higiénico transparente que impida la propagación de errores hacia las fases avanzadas del proyecto, el cuaderno se organiza de forma secuencial en seis dimensiones críticas:

1. **Configuración del Entorno de Trabajo:**
   * Inicialización del entorno local y carga de las librerías del núcleo de datos (`pandas`, `numpy`, `json`).
   * Configuración de rutas relativas universales mediante `pathlib` para blindar la portabilidad del repositorio dentro de la carpeta raíz del equipo.
2. **Carga Atómica Segura (Deserialización):**
   * Lectura e importación individualizada desde el disco hacia la memoria RAM de los 8 archivos JSON que componen el ecosistema informativo.
   * Aislamiento de las estructuras en DataFrames crudos independientes para iniciar el análisis sin alteraciones cruzadas.
3. **Auditoría Cuantitativa (Perfilado Numérico):**
   * Escaneo sistemático de los atributos continuos y discretos.
   * Cuantificación absoluta de valores nulos del sistema (`NaN`), cálculo matemático de rangos límite (mínimos y máximos) y detección temprana de registros negativos atípicos que violen la naturaleza del campo.
4. **Auditoría Cualitativa (Perfilado de Texto):**
   * Evaluación forense sobre columnas de tipo *string* o cadenas de texto libre.
   * Análisis descriptivo de longitudes, detección de contaminación por espacios y rastreo de anomalías de interfaz de usuario como cadenas vacías (`""`) o la inyección encubierta del valor centinela `"-1"`.
5. **Matriz Consolidada de Salud del Dato (Data Quality Matrix):**
   * Fusión de todas las métricas de control en una única tabla resumen por cada entidad analizada.
   * Este reporte expone visualmente los registros totales, la cardinalidad de valores únicos, el volumen de nulos, la presencia de vacíos y la tasa de anomalías detectadas, sirviendo como el mapa de ruta definitivo para las tareas de limpieza.
6. **Exportación y Cierre de Contrato:**
   * Formateo final de las estructuras y almacenamiento ordenado de los sets de datos.
   * Los archivos quedan preparados en disco para alimentar dos ramas del pipeline: `datos_3FN/` para el **Notebook 2 (Ingesta en MySQL)** y `datos_1FN/` (Parquet) para el **Notebook 3 (EDA)**.

---

## 🏁 **Garantías de Salida (El Compromiso de la Fase 1)**

Al concluir la ejecución de este cuaderno, el proyecto asegura:
* **Orden de Repositorio:** Eliminación absoluta de archivos huérfanos o desorganizados en la raíz, centralizando el origen estrictamente en `./datos_raw/`.
* **Línea Base Documentada:** Una radiografía transparente que expone detalladamente el grado de contaminación inicial del dataset.
* **Desacoplamiento Estructural:** Las 8 entidades quedan diagnosticadas y estructuradas, lo que permite la carga en MySQL (NB2) mediante DDL (definición de tablas) y DML (inserción de filas) como el análisis exploratorio (NB3), cada uno conociendo con precisión el tipo y la calidad de los datos de partida.

# **1. Pipeline de ingesta: configuración y deserialización de fuentes (JSON)**

## 1.1. Configuración del entorno e importación de dependencias base

In [176]:
import os                                                               # Rutas y comprobación de archivos en disco
import json                                                             # Lectura y parseo de los ficheros JSON de origen
import numpy as np                                                      # Soporte para operaciones matemáticas
import pandas as pd                                                     # Manipulación de datos y DataFrames
from pathlib import Path                                                # Gestión universal de rutas
from IPython.display import Markdown, display                           # Notas dinámicas en markdown (memoria, ahorros)

print("✅ Entorno del Notebook 1 configurado correctamente.")

✅ Entorno del Notebook 1 configurado correctamente.


## 1.2. Detección dinámica de infraestructura y motor de ingesta universal

In [177]:
# 1. Definición estricta de la ruta local de origen (Capa Raw / Datos Brutos)
RUTA_RAW = Path('.') / 'datos_raw'

print("✅ Entorno configurado: Local (Proyecto Júpiter Final).")
print(f"Verificando acceso a la carpeta de origen: '{RUTA_RAW.resolve()}'\n")

# 2. Motor de ingesta seguro exclusivo para arquitectura local
def cargar_json_seguro(nombre_archivo):
    """Verifica la existencia del archivo dentro de la subcarpeta 'datos_raw' y previene errores de ejecución."""
    ruta = RUTA_RAW / nombre_archivo
    
    if not ruta.exists():
        raise FileNotFoundError(
            f"\n❌ ERROR CRÍTICO: No se encuentra el archivo '{nombre_archivo}'.\n"
            f"Ruta exacta intentada por el sistema: '{ruta.resolve()}'\n"
            f"SOLUCCIÓN: Asegúrate de que el archivo '{nombre_archivo}' esté físicamente metido dentro de la carpeta 'datos_raw'."
        )
        
    with open(ruta, 'r', encoding='utf-8') as f:
        return json.load(f)

# 3. Deserialización atómica desde el disco local a la memoria RAM
try:
    tabla_info             = cargar_json_seguro('info.json')
    tabla_tipo             = cargar_json_seguro('tipo.json')
    tabla_fake             = cargar_json_seguro('fake.json')
    tabla_tiempo           = cargar_json_seguro('tiempo.json')
    tabla_contenido        = cargar_json_seguro('contenido.json')
    tabla_metadatos        = cargar_json_seguro('metadatos.json')
    tabla_pais_idioma      = cargar_json_seguro('pais_idioma.json')
    tabla_compartir_tiempo = cargar_json_seguro('compartir_tiempo.json')

    print("✅ INGESTA COMPLETADA: Los 8 archivos JSON se han cargado en memoria exitosamente.")
    print("Estructuras cargadas y listas para la fase de normalización y análisis.")

except Exception as e:
    print(f"\n❌ Fallo en la inicialización del pipeline: {e}")

✅ Entorno configurado: Local (Proyecto Júpiter Final).
Verificando acceso a la carpeta de origen: 'C:\Users\<usuario>\Desktop\PontIA\Curso\6. Proyecto Jupiter\0. Proyecto Fake News\Proyecto_Jupiter_Final\datos_raw'

✅ INGESTA COMPLETADA: Los 8 archivos JSON se han cargado en memoria exitosamente.
Estructuras cargadas y listas para la fase de normalización y análisis.


## 1.3. Medición de memoria del downcasting

Función auxiliar que captura la memoria de las columnas numéricas antes de castearlas, para calcular después el ahorro global del downcasting (sección 2.9). El texto libre queda fuera, por no ser objetivo de downcasting.

In [178]:
_mem_ini = {}

COLS_CAST = {
    "info": ["t_id"],
    "tipo": ["t_id"],
    "fake": ["t_id", "fake"],
    "tiempo": ["t_id", "tiempo"],
    "contenido": ["t_id"],
    "metadatos": ["t_id", "visitas", "compartir", "duracion", "favorito"],
    "pais_idioma": ["t_id"],
    "compartir_tiempo": ["t_id", "compartir_tiempo"],
}

def _kb_cols(df, cols):
    return df[cols].memory_usage(deep=True).drop('Index', errors='ignore').sum() / 1024

def memoria_inicial(df, etiqueta):             # ANTES de castear (columnas aún en su tipo original)
    _mem_ini[etiqueta] = _kb_cols(df, COLS_CAST[etiqueta])
    display(Markdown(f"**{etiqueta} - columnas a castear (inicial):** {_mem_ini[etiqueta]:.1f} KB."))

# **2.Inicialización de los frameworks analíticos y auditorías de sus estructuras**

## **2.1. ⊞ Tabla info**

### **2.1.1. Diagnóstico de integridad y arquitectura del esquema**

In [179]:
# Auditoría técnica de la entidad
tipo_obj_tabla_info = type(tabla_info)                                                                          # Identificación de la clase del objeto
cardinalidad_tabla_info = len(tabla_info)                                                                       # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla info | Clase: {tipo_obj_tabla_info} | Cardinalidad: {cardinalidad_tabla_info}")          # Reporte de consistencia y validación de carga

Entidad: Tabla info | Clase: <class 'dict'> | Cardinalidad: 2


In [180]:
# Inspección del esquema de claves (Keys).
tabla_info.keys()

dict_keys(['autor', 'fuente'])

In [181]:
# Identificación de dimensiones de clasificación.
tabla_info

{'autor': [[8476, 'Marcus Liu'],
  [10294, 'Lily Wang'],
  [3608, 'Nora Patel'],
  [10142, 'Avery Kim'],
  [875, 'Daniel Patel'],
  [6903, 'Noah Khan'],
  [7341, 'Victoria Singh'],
  [95, 'Owen Rodriguez'],
  [4869, 'Isabella Nguyen'],
  [2909, 'Ella Sharma'],
  [1357, 'Ethan Hayes'],
  [988, 'Amelia Rivera'],
  [7041, 'Alexander Lee'],
  [7623, 'Daniel Patel'],
  [1571, 'Lily Wang'],
  [4739, 'Madison Wong'],
  [7737, 'Lucas Kim'],
  [8716, 'Nora Patel'],
  [3304, 'Alexander Lee'],
  [3078, 'Owen Rodriguez'],
  [2517, -1],
  [10348, 'Lucas Kim'],
  [778, 'Grace Liu'],
  [3300, 'Emily Parker'],
  [6155, -1],
  [636, 'Madison Wong'],
  [755, 'Madison Wong'],
  [626, 'Avery Kim'],
  [691, ''],
  [5743, 'Victoria Singh'],
  [1787, 'Ella Sharma'],
  [7808, 'Jacob Gupta'],
  [6484, -1],
  [7385, 'Xavier Chang'],
  [9016, 'Xavier Chang'],
  [9324, 'Samantha Chen'],
  [8983, -1],
  [8965, 'Jacob Gupta'],
  [5580, 'Sophie Chen'],
  [9757, 'Caleb Shah'],
  [1967, -1],
  [431, 'Lucas Kim'],
  [5

### **2.1.2. Transformación a estructura tabular y normalización del esquema**

In [182]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_info = pd.json_normalize(tabla_info)

#### **2.1.2.1. Diagnóstico de anomalías estructurales: auditoría de granularidad, anidamiento y dispersión dimensional**

In [183]:
# Diagnóstico de integridad estructural: Detección de registros anidados (Nested Data Structures).
df_tabla_info

,autor,fuente
0,"[[8476, Marcus Liu], [10294, Lily Wang], [3608...","[[8476, The Daily Dispatch], [10294, Insider I..."


**Nota:** Identificación de **anomalía de granularidad**: los **registros carecen de atomicidad** al presentarse como **estructuras anidadas** dentro de una única instancia, violando la Primera Forma Normal (1FN).

#### **2.1.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [184]:
# Aislamiento de vectores serializados para segmentación de atributos
autor_data = tabla_info['autor']
fuente_data = tabla_info['fuente']

# Transformación atributos multivaluados (listas) en entidades escalares independientes (1FN).
df_autor_flat = pd.DataFrame(autor_data, columns=['t_id', 'autor'])                                     # Mapeo atómico del identificador y la entidad 'autor'
df_fuente_flat = pd.DataFrame(fuente_data, columns=['t_id', 'fuente'])                                  # Mapeo atómico del identificador y la entidad 'fuente'

# Consolidación de entidades mediante JOIN y validación de integridad referencial (1:1)
df_tabla_info_limpio = df_autor_flat.merge(df_fuente_flat, on='t_id', validate='one_to_one')            # Fusión de series garantizando la paridad de registros

# Reestructuración final a esquema tabular y jerarquización de atributos
df_tabla_info_limpio = df_tabla_info_limpio[['t_id', 'autor', 'fuente']]                                # Alineación de columnas según el diseño de tabla final (1FN)



In [185]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_info_limpio.head()

,t_id,autor,fuente
0,8476,Marcus Liu,The Daily Dispatch
1,10294,Lily Wang,Insider Insights
2,3608,Nora Patel,Daily Beacon
3,10142,Avery Kim,The Pulse
4,875,Daniel Patel,Morning Star


### **2.1.3. Auditoría de tabla info y validación dimensional (2FN)**

In [186]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_info_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_info_limpio, "info")
df_tabla_info_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   int64 
 1   autor   6069 non-null   object
 2   fuente  6335 non-null   object
dtypes: int64(1), object(2)
memory usage: 879.6 KB


**info - columnas a castear (inicial):** 49.5 KB.

(6335, 3)

**Nota:** Resultado inspección **tabla 'info'**: el campo **t_id** presenta formato numérico (**int64**). En cambio, **autor** y **fuente** presentan formato **object** requieren **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.1.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.1.3.1.1. Pre-Casting y validación de Dominio**

In [187]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla info' es:", df_tabla_info_limpio['t_id'].min())
print("El valor máximo del campo 't_id' en la 'tabla info' es:", df_tabla_info_limpio['t_id'].max())

El valor mínimo del campo 't_id' en la 'tabla info' es: 2
El valor máximo del campo 't_id' en la 'tabla info' es: 10557


**Nota:** Valores estrictamente positivos sin riesgo de overflow para la capacidad de uint16.

##### **2.1.3.1.2. Casting del campo**

In [188]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_info_limpio['t_id'] = df_tabla_info_limpio['t_id'].astype('uint16')                                        # Aplicación de downcasting + valores estrictamente positivos
df_tabla_info_limpio.info(memory_usage='deep')                                                                      # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   autor   6069 non-null   object
 2   fuente  6335 non-null   object
dtypes: object(2), uint16(1)
memory usage: 842.5 KB


##### **2.1.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [189]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la 'tabla info': {len(df_tabla_info_limpio)}")
print(f"Valores únicos del campo 't_id' en la 'tabla info': {df_tabla_info_limpio['t_id'].nunique()}")                      # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la 'tabla info': {df_tabla_info_limpio['t_id'].isna().sum()}")           # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)

Total de registros de la 'tabla info': 6335
Valores únicos del campo 't_id' en la 'tabla info': 6335
Nulos oficiales (isna) del campo 't_id' en la 'tabla info': 0


**Nota: conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de **'t_id'** de la **tabla 'info', se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.1.3.2. Atributo 'autor': auditoría, downcasting y unicidad visual.**

##### **2.1.3.2.1. Pre-Casting y validación de Dominio**

In [190]:
# Preparación y diagnóstico de tipos de datos
comp_campo_autor = df_tabla_info_limpio["autor"]                                                                            # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_autor = comp_campo_autor.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()         # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_autor.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                   # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 'autor':")                                                                           # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_autor.to_string(index=False))                                                                       # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 'autor':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 5754
         int                  315
    NoneType                  266


**Nota:** Hay **presencia de tipos mixtos** en el campo **'autor'** tipo de dato object. Se audita cada los datos para ver como proceder.

##### **a) Límite del campo**

In [191]:
# Muestra de valores únicos
mask_string_campo_autor = df_tabla_info_limpio["autor"].apply(lambda x: isinstance(x, str))
string_puros_campo_autor = df_tabla_info_limpio.loc[mask_string_campo_autor, "autor"]
valores_unicos_campo_autor = string_puros_campo_autor.nunique()

print(f"Valores únicos en el campo 'autor': {valores_unicos_campo_autor}")

Valores únicos en el campo 'autor': 35


##### **Resumen de frecuencia absoluta**

In [192]:
tabla_distribucion_campo_autor = df_tabla_info_limpio['autor'].value_counts(dropna=False).reset_index()     # Agregación por categorías incluyendo todos los tipos de datos
tabla_distribucion_campo_autor.columns = ['VALORES ÚNICOS CAMPO AUTOR', 'Frecuencia Absoluta']              # Normalización nomenclatura
total_campo_autor = tabla_distribucion_campo_autor['Frecuencia Absoluta'].sum()                             # Cálculo frecuencia absoluta

tabla_distribucion_campo_autor.index = tabla_distribucion_campo_autor.index + 1
tabla_distribucion_campo_autor.loc[""] = ['TOTAL', total_campo_autor]
tabla_distribucion_campo_autor

,VALORES ÚNICOS CAMPO AUTOR,Frecuencia Absoluta
1,,338
2,-1,315
3,None,266
4,Owen Rodriguez,186
5,Isabella Nguyen,178
6,Alexander Lee,178
7,Michael Sharma,173
8,Amelia Rivera,169
9,William Kumar,168
10,Chloe Nguyen,166


**Nota:** Para el **campo 'autor'** se detectan **37 categorías únicas** de las cuales **3 agrupan valores anómalos**: "vacíos", "-1" y "None".  Para evitar redundancia de datos este **texto se extraerá a una tabla catálogo (dimensión)**, dejando únicamente su ID (3FN).

In [193]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_autor = string_puros_campo_autor.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_autor = longitudes_campo_autor.min()
max_caracteres_campo_autor = longitudes_campo_autor.max()

# Extracción de las registros de los limites
string_mas_corto_campo_autor = string_puros_campo_autor.loc[longitudes_campo_autor.idxmin()]
string_mas_largo_campo_autor = string_puros_campo_autor.loc[longitudes_campo_autor.idxmax()]

# Reporte
print("Auditoría de límites de longitud del campo 'autor' (Categorías):")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_autor)} caracteres. Registro '{string_mas_corto_campo_autor}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_autor)} caracteres. Registro '{string_mas_largo_campo_autor}'")

Auditoría de límites de longitud del campo 'autor' (Categorías):
-----------------------------------------------------------------
Límite inferior (Min): 0 caracteres. Registro ''
Límite superior (Max): 15 caracteres. Registro 'Isabella Nguyen'


##### **2.1.3.2.2. Casting del campo**

**Nota:** Se determina que la optimización del tipo de dato para el atributo categórico **'autor' (definido como VARCHAR(30))** se ejecutará en la fase de persistencia en **MySQL**. Debido a que en el entorno de desarrollo (Pandas) los strings se gestionan como objetos de memoria, la segregación y el casting final se realizarán mediante la creación de la **Tabla Maestra [AUTOR_ID vs. AUTOR]**. Esta estrategia asegura la integridad referencial y el cumplimiento estricto de la Tercera Forma Normal **(3FN)**, optimizando la eficiencia de almacenamiento y evitando la redundancia de datos.


#### **2.1.3.3. Atributo 'fuente': auditoría, downcasting y unicidad visual.**

##### **2.1.3.3.1. Pre-Casting y validación de Dominio**

In [194]:
# Preparación y diagnóstico de tipos de datos
comp_campo_fuente = df_tabla_info_limpio["fuente"]                                                                        # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_fuente = comp_campo_fuente.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_fuente.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 'fuente':")                                                                        # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_fuente.to_string(index=False))                                                                    # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 'fuente':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'fuente'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [195]:
# Muestra de valores únicos
mask_string_campo_fuente = df_tabla_info_limpio["fuente"].apply(lambda x: isinstance(x, str))
string_puros_campo_fuente = df_tabla_info_limpio.loc[mask_string_campo_fuente, "fuente"]
valores_unicos_campo_fuente = string_puros_campo_fuente.nunique()

print(f"Valores únicos en el campo 'fuente': {valores_unicos_campo_fuente}")

Valores únicos en el campo 'fuente': 33


##### **Resumen de frecuencia absoluta**

In [196]:
tabla_distribucion_campo_fuente = df_tabla_info_limpio['fuente'].value_counts(dropna=False).reset_index()           # Agregación por categorías incluyendo todos los tipos de datos
tabla_distribucion_campo_fuente.columns = ['VALORES ÚNICOS CAMPO FUENTE', 'Frecuencia Absoluta']                    # Normalización nomenclatura
total_campo_fuente = tabla_distribucion_campo_fuente['Frecuencia Absoluta'].sum()                                   # Cálculo frecuencia absoluta

tabla_distribucion_campo_fuente.index = tabla_distribucion_campo_fuente.index + 1
tabla_distribucion_campo_fuente.loc[""] = ['TOTAL', total_campo_fuente]
tabla_distribucion_campo_fuente

,VALORES ÚNICOS CAMPO FUENTE,Frecuencia Absoluta
1,The Insight,384
2,Sunrise Chronicle,215
3,Times Today,212
4,Metro News,211
5,The Daily Dispatch,198
6,Press Pulse,196
7,Voice of the Nation,196
8,Newsflash,192
9,Evening Echo,192
10,Urban News,190


**Nota:** Para el **campo 'fuente'** se detectan **33 categorías únicas**.  Para evitar redundancia de datos este **fuente se extraerá a una tabla catálogo (dimensión)**, dejando únicamente su ID (3FN).

In [197]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_fuente = string_puros_campo_fuente.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_fuente = longitudes_campo_fuente.min()
max_caracteres_campo_fuente = longitudes_campo_fuente.max()

# Extracción de las registros de los limites
string_mas_corto_campo_fuente = string_puros_campo_fuente.loc[longitudes_campo_fuente.idxmin()]
string_mas_largo_campo_fuente = string_puros_campo_fuente.loc[longitudes_campo_fuente.idxmax()]

# Reporte
print("Auditoría de límites de longitud del campo 'fuente' (Categorías):")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_fuente)} caracteres. Registro '{string_mas_corto_campo_fuente}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_fuente)} caracteres. Registro '{string_mas_largo_campo_fuente}'")

Auditoría de límites de longitud del campo 'fuente' (Categorías):
-----------------------------------------------------------------
Límite inferior (Min): 8 caracteres. Registro 'Newsline'
Límite superior (Max): 19 caracteres. Registro 'Voice of the Nation'


##### **2.1.3.3.2. Casting del campo**

**Nota:** Se determina que la optimización del tipo de dato para el atributo categórico **'fuente' (definido como VARCHAR(30))** se ejecutará en la fase de persistencia en **MySQL**. Debido a que en el entorno de desarrollo (Pandas) los strings se gestionan como objetos de memoria, la segregación y el casting final se realizarán mediante la creación de la **Tabla Maestra [FUENTE_ID vs. FUENTE]**. Esta estrategia asegura la integridad referencial y el cumplimiento estricto de la Tercera Forma Normal **(3FN)**, optimizando la eficiencia de almacenamiento y evitando la redundancia de datos.


## **2.2. ⊞ Tabla tipo**

### **2.2.1. Diagnóstico de integridad y arquitectura del esquema**

In [198]:
# Auditoría técnica de la entidad
tipo_obj_tabla_tipo = type(tabla_tipo)                                                                      # Identificación de la clase del objeto
cardinalidad_tabla_tipo = len(tabla_tipo)                                                                   # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla tipo | Clase: {tipo_obj_tabla_tipo} | Cardinalidad: {cardinalidad_tabla_tipo}")      # Reporte de consistencia y validación de carga

Entidad: Tabla tipo | Clase: <class 'dict'> | Cardinalidad: 21


In [199]:
# Inspección del esquema de claves (Keys).
tabla_tipo.keys()

dict_keys(['Columna', 'Artículo de opinión', 'Carta al editor', 'Entrevista', 'Reportaje', 'Crónica', 'Noticia de última hora', 'Editorial', 'Reseña', 'Crítica', 'Análisis', 'Entrevista exclusiva', 'Columna de humor', 'Noticia local', 'Noticia internacional', 'Artículo de investigación', 'Entrevista a fondo', 'Entrevista en profundidad', 'Opinión del lector', 'Reportaje especial', 'Noticia destacada'])

In [200]:
# Identificación de dimensiones de clasificación.
tabla_tipo

{'Columna': [3608,
  4869,
  139,
  3424,
  1718,
  3635,
  6227,
  2232,
  4114,
  186,
  4714,
  4929,
  9265,
  1109,
  9196,
  2678,
  5035,
  6804,
  266,
  4305,
  1118,
  4750,
  1675,
  8902,
  5760,
  9565,
  1883,
  1532,
  612,
  10270,
  10444,
  6880,
  7667,
  4178,
  8525,
  10372,
  2241,
  1204,
  2216,
  701,
  2092,
  6055,
  998,
  8879,
  9009,
  8496,
  23,
  5233,
  2935,
  597,
  3115,
  6279,
  7769,
  5064,
  929,
  2994,
  351,
  7559,
  654,
  5991,
  5214,
  2746,
  5416,
  7161,
  2171,
  5948,
  510,
  7255,
  4486,
  4471,
  7785,
  734,
  6566,
  7411,
  8206,
  777,
  7720,
  4983,
  9424,
  5326,
  401,
  385,
  10109,
  1116,
  4586,
  1140,
  362,
  4091,
  1051,
  8372,
  6032,
  6359,
  3601,
  3005,
  877,
  169,
  7028,
  1269,
  916,
  2761,
  8325,
  2203,
  183,
  342,
  8109,
  3418,
  6097,
  10187,
  8942,
  355,
  1588,
  8513,
  5151,
  5250,
  5154,
  5915,
  1778,
  1808,
  2531,
  1028,
  6067,
  1949,
  4840,
  354,
  3733,
  3175,
 

### **2.2.2. Transformación a estructura tabular y normalización del esquema**

In [201]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_tipo = pd.json_normalize(tabla_tipo)

#### **2.2.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [202]:
# Identificación de registros no atómicos: estructura 1:N detectada en el esquema.
df_tabla_tipo

,Columna,Artículo de opinión,Carta al editor,Entrevista,Reportaje,Crónica,Noticia de última hora,Editorial,Reseña,Crítica,...,Entrevista exclusiva,Columna de humor,Noticia local,Noticia internacional,Artículo de investigación,Entrevista a fondo,Entrevista en profundidad,Opinión del lector,Reportaje especial,Noticia destacada
0,"[3608, 4869, 139, 3424, 1718, 3635, 6227, 2232...","[4739, 3300, 5931, 3648, 6815, 2432, 1770, 258...","[1787, 7455, 8177, 6763, 3004, 1329, 3282, 461...","[9016, 587, 9403, 3627, 3363, 8988, 5040, 8767...","[3027, 5530, 4500, 8357, 172, 7757, 1255, 658,...","[8476, 3255, 10427, 6809, 2802, 3138, 3700, 38...","[10142, 7341, 7041, 7623, 7737, 10348, 6155, 9...","[6903, 2909, 8965, 5521, 3343, 1524, 2546, 520...","[10294, 8716, 3304, 626, 1834, 1475, 4979, 891...","[4377, 4348, 2832, 4294, 9413, 6247, 4354, 975...",...,"[2517, 4410, 1360, 4933, 10490, 2596, 6025, 21...","[5743, 8983, 2974, 6212, 3602, 7892, 10183, 38...","[3078, 7385, 1084, 2199, 677, 7154, 1934, 6698...","[778, 5580, 5224, 1506, 143, 3364, 3512, 9468,...","[9324, 596, 930, 7730, 127, 8682, 7256, 5297, ...","[875, 755, 1967, 5955, 5937, 1178, 917, 1661, ...","[431, 7540, 1641, 7680, 9398, 7818, 8322, 9447...","[988, 6484, 1775, 3854, 10232, 4943, 10092, 84...","[9360, 4134, 7513, 421, 6851, 6407, 5307, 4217...","[1571, 636, 7808, 2669, 1569, 7099, 9295, 3008..."


**Nota:** Identificación de anomalía estructural: presencia de **colecciones anidadas**. El esquema actual colapsa múltiples entidades en un solo registro, **impidiendo la atomicidad del registro** (1FN).

#### **2.2.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [203]:
# Se transforma la granularidad de colecciones (1:N) a instancias atómicas (1:1)
filas_tabla_tipo = []                                                 # Inicialización del contenedor para la estructura flat (plana).
for tipo, lista_ids_tabla_tipo in df_tabla_tipo.iloc[0].items():      # Iteración sobre el esquema de atributos y sus colecciones asociadas.
   for t_id in lista_ids_tabla_tipo:                                  # 'Unpacking' de elementos para la generación de registros atómicos.
      filas_tabla_tipo.append({
              't_id': t_id,                                           # Mapeo del identificador único (Clave Primaria)
              'tipo': tipo                                            # Mapeo del atributo de dimensión (Categoría)
        })
df_tabla_tipo_limpio = pd.DataFrame(filas_tabla_tipo)                 # Instanciación del DataFrame final con integridad referencial 1:1.

In [204]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_tipo_limpio.head(5)

,t_id,tipo
0,3608,Columna
1,4869,Columna
2,139,Columna
3,3424,Columna
4,1718,Columna


**Nota:** Se identifica **t_id** como clave **candidata a PK** por su variabilidad frente a la homogeneidad de tipo. Su ratificación como **PK** depende de validar unicidad y no nulidad para garantizar la **1FN**.

### **2.2.3. Auditoría de tabla tipo y validación dimensional (2FN)**

In [205]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_tipo_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_tipo_limpio, "tipo")
df_tabla_tipo_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6029 entries, 0 to 6028
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6029 non-null   int64 
 1   tipo    6029 non-null   object
dtypes: int64(1), object(1)
memory usage: 548.8 KB


**tipo - columnas a castear (inicial):** 47.1 KB.

(6029, 2)

**Nota:** Resultado inspección **tabla 'tipo'**: el campo **t_id** presenta el formato numérico (**int64**). El campo **tipo** (**object**) requiere **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.2.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.2.3.1.1. Pre-Casting y validación de Dominio**

In [206]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla tipo' es:", df_tabla_tipo_limpio['t_id'].min())
print("El valor máximo del campo 't_id' en la 'tabla tipo' es:", df_tabla_tipo_limpio['t_id'].max())

El valor mínimo del campo 't_id' en la 'tabla tipo' es: 2
El valor máximo del campo 't_id' en la 'tabla tipo' es: 10557


**Nota:** Valores estrictamente positivos sin riesgo de overflow para la capacidad de uint16.

##### **2.2.3.1.2. Casting del campo**

In [207]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_tipo_limpio['t_id'] = df_tabla_tipo_limpio['t_id'].astype('uint16')                                        # Aplicación de downcasting + valores estrictamente positivos
df_tabla_tipo_limpio.info(memory_usage='deep')                                                                      # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6029 entries, 0 to 6028
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6029 non-null   uint16
 1   tipo    6029 non-null   object
dtypes: object(1), uint16(1)
memory usage: 513.4 KB


##### **2.2.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [208]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la 'tabla tipo': {len(df_tabla_tipo_limpio)}")
print(f"Valores únicos del campo 't_id' en la 'tabla tipo': {df_tabla_tipo_limpio['t_id'].nunique()}")                      # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la 'tabla tipo': {df_tabla_tipo_limpio['t_id'].isna().sum()}")           # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)

Total de registros de la 'tabla tipo': 6029
Valores únicos del campo 't_id' en la 'tabla tipo': 6029
Nulos oficiales (isna) del campo 't_id' en la 'tabla tipo': 0


**Nota: conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de **'t_id'** de la **tabla 'tipo', se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.2.3.2. Atributo 'tipo': auditoría, downcasting y unicidad visual.**

##### **2.2.3.2.1. Pre-Casting y validación de Dominio**

In [209]:
# Preparación y diagnóstico de tipos de datos
comp_campo_tipo = df_tabla_tipo_limpio["tipo"]                                                                        # Extracción del campo para su análisis individual
tabla_tipos_campo_tipo= comp_campo_tipo.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()      # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_tipo.columns = ['Tipo de dato', 'Frecuencia absoluta']                                              # Asignación cabeceras a las columnas

print("Tipo objetos Python en 'tipo':")                                                                               # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_tipo.to_string(index=False))                                                                  # Impresión con el formato tabular limpio

Tipo objetos Python en 'tipo':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6029


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'tipo'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [210]:
# Muestra de valores únicos
mask_string_campo_tipo = df_tabla_tipo_limpio["tipo"].apply(lambda x: isinstance(x, str))
string_puros_campo_tipo = df_tabla_tipo_limpio.loc[mask_string_campo_tipo, "tipo"]
valores_unicos_campo_tipo= string_puros_campo_tipo.nunique()

print(f"Valores únicos en el campo 'tipo': {valores_unicos_campo_tipo}")

Valores únicos en el campo 'tipo': 21


##### **Resumen de frecuencia absoluta**

In [211]:
tabla_distribucion_campo_tipo = df_tabla_tipo_limpio['tipo'].value_counts().reset_index()         # Agregación por categorías
tabla_distribucion_campo_tipo.columns = ['VALORES ÚNICOS CAMPO TIPO', 'Frecuencia Absoluta']      # Normalización nomenclatura
total_campo_tipo = tabla_distribucion_campo_tipo['Frecuencia Absoluta'].sum()                     # Cálculo frecuencia absoluta

tabla_distribucion_campo_tipo.index = tabla_distribucion_campo_tipo.index + 1
tabla_distribucion_campo_tipo.loc[""] = ['TOTAL', total_campo_tipo]
tabla_distribucion_campo_tipo

,VALORES ÚNICOS CAMPO TIPO,Frecuencia Absoluta
1,Noticia de última hora,1401
2,Crítica,268
3,Crónica,254
4,Reportaje,249
5,Entrevista a fondo,248
6,Noticia destacada,238
7,Carta al editor,238
8,Columna de humor,236
9,Noticia internacional,236
10,Reseña,231


**Nota:** Para el **campo 'tipo'** se detectan **21 categorías únicas**. Para evitar redundancia de datos este **texto se extraerá a una tabla catálogo (dimensión)**, dejando únicamente su ID (3FN).

In [212]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_tipo = string_puros_campo_tipo.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_tipo = longitudes_campo_tipo.min()
max_caracteres_campo_tipo = longitudes_campo_tipo.max()

# Extracción de las registros de los limites
string_mas_corto_campo_tipo = string_puros_campo_tipo.loc[longitudes_campo_tipo.idxmin()]
string_mas_largo_campo_tipo = string_puros_campo_tipo.loc[longitudes_campo_tipo.idxmax()]

# Reporte
print("Auditoría de límites de longitud del campo 'tipo' (Categorías):")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_tipo)} caracteres. Registro '{string_mas_corto_campo_tipo}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_tipo)} caracteres. Registro '{string_mas_largo_campo_tipo}'")

Auditoría de límites de longitud del campo 'tipo' (Categorías):
-----------------------------------------------------------------
Límite inferior (Min): 6 caracteres. Registro 'Reseña'
Límite superior (Max): 25 caracteres. Registro 'Artículo de investigación'


**Nota:** 21 categorías. Ninguna anómala. Se confirma que el tipo de noticia no es PK y que depende de t_id (2FN)

##### **2.2.3.2.2. Casting del campo**

**Nota:** Se determina que la optimización del tipo de dato para el atributo categórico **'tipo' (definido como VARCHAR(30))** se ejecutará en la fase de persistencia en **MySQL**. Debido a que en el entorno de desarrollo (Pandas) los strings se gestionan como objetos de memoria, la segregación y el casting final se realizarán mediante la creación de la **Tabla Maestra [TIPO_ID vs. TIPO]**. Esta estrategia asegura la integridad referencial y el cumplimiento estricto de la Tercera Forma Normal **(3FN)**, optimizando la eficiencia de almacenamiento y evitando la redundancia de datos.


## **2.3. ⊞ Tabla fake**

### **2.3.1. Diagnóstico de integridad y arquitectura del esquema**

In [213]:
# Auditoría técnica de la entidad
tipo_obj_tabla_fake = type(tabla_fake)                                                                      # Identificación de la clase del objeto
cardinalidad_tabla_fake = len(tabla_fake)                                                                   # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla fake | Clase: {tipo_obj_tabla_fake} | Cardinalidad: {cardinalidad_tabla_fake}")      # Reporte de consistencia y validación de carga

Entidad: Tabla fake | Clase: <class 'dict'> | Cardinalidad: 6335


In [214]:
# Inspección del esquema de claves (Keys).
tabla_fake.keys()

dict_keys(['8476', '10294', '3608', '10142', '875', '6903', '7341', '95', '4869', '2909', '1357', '988', '7041', '7623', '1571', '4739', '7737', '8716', '3304', '3078', '2517', '10348', '778', '3300', '6155', '636', '755', '626', '691', '5743', '1787', '7808', '6484', '7385', '9016', '9324', '8983', '8965', '5580', '9757', '1967', '431', '5955', '7455', '5224', '7793', '2777', '587', '9403', '5931', '3027', '4377', '5937', '7277', '5521', '3627', '9360', '909', '6646', '1522', '7540', '1834', '1775', '8212', '3363', '1084', '4410', '2669', '3648', '6197', '596', '6061', '1569', '1178', '9545', '1506', '1641', '139', '7744', '8177', '4134', '917', '3854', '7049', '3343', '4348', '3424', '1718', '143', '930', '1661', '7801', '10232', '1360', '3635', '8290', '6227', '3255', '3177', '2832', '4294', '5316', '1524', '2974', '3364', '4933', '5530', '7680', '10490', '9413', '4880', '3512', '8685', '6763', '7730', '2546', '2596', '7362', '7513', '1475', '2232', '6247', '127', '6025', '10122', '

In [215]:
# Identificación de dimensiones de clasificación.
tabla_fake

{'8476': 'FAKE',
 '10294': 'FAKE',
 '3608': 'REAL',
 '10142': 'FAKE',
 '875': 'REAL',
 '6903': 'FAKE',
 '7341': 'FAKE',
 '95': 'REAL',
 '4869': 'REAL',
 '2909': 'REAL',
 '1357': 'REAL',
 '988': 'REAL',
 '7041': 'FAKE',
 '7623': 'FAKE',
 '1571': 'REAL',
 '4739': 'REAL',
 '7737': 'FAKE',
 '8716': 'FAKE',
 '3304': 'REAL',
 '3078': 'REAL',
 '2517': 'REAL',
 '10348': 'FAKE',
 '778': 'REAL',
 '3300': 'REAL',
 '6155': 'FAKE',
 '636': 'REAL',
 '755': 'REAL',
 '626': 'REAL',
 '691': 'REAL',
 '5743': 'FAKE',
 '1787': 'REAL',
 '7808': 'FAKE',
 '6484': 'FAKE',
 '7385': 'FAKE',
 '9016': 'FAKE',
 '9324': 'FAKE',
 '8983': 'FAKE',
 '8965': 'FAKE',
 '5580': 'FAKE',
 '9757': 'FAKE',
 '1967': 'REAL',
 '431': 'REAL',
 '5955': 'FAKE',
 '7455': 'FAKE',
 '5224': 'REAL',
 '7793': 'FAKE',
 '2777': 'REAL',
 '587': 'REAL',
 '9403': 'FAKE',
 '5931': 'FAKE',
 '3027': 'REAL',
 '4377': 'REAL',
 '5937': 'FAKE',
 '7277': 'FAKE',
 '5521': 'FAKE',
 '3627': 'REAL',
 '9360': 'FAKE',
 '909': 'REAL',
 '6646': 'FAKE',
 '1522

### **2.3.2. Transformación a estructura tabular y normalización del esquema**

In [216]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_fake = pd.json_normalize(tabla_fake)

#### **2.3.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [217]:
# Diagnóstico de integridad estructural: Detección de esquema horizontal (Wide Format) no relacional.
df_tabla_fake

,8476,10294,3608,10142,875,6903,7341,95,4869,2909,...,8411,6143,3262,9337,8737,4490,8062,8622,4021,4330
0,FAKE,FAKE,REAL,FAKE,REAL,FAKE,FAKE,REAL,REAL,REAL,...,FAKE,FAKE,REAL,FAKE,FAKE,REAL,FAKE,FAKE,REAL,REAL


**Nota:** Estructura *wide format* con entidades transpuestas. Aunque existe atomicidad (1FN), la dispersión de t_id en columnas impide el procesamiento vectorial eficiente.

#### **2.3.2.2. Reestructuración lógica**

In [218]:
# Transposición de entidades y tipado de campos
df_tabla_fake_limpio = df_tabla_fake.T.reset_index()      # Pivotado de entidades (.T)
df_tabla_fake_limpio.columns = ['t_id', 'fake']           # Definición de cabeceras

In [219]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_fake_limpio.head(5)

,t_id,fake
0,8476,FAKE
1,10294,FAKE
2,3608,REAL
3,10142,FAKE
4,875,REAL


**Nota:** Se identifica **t_id** como clave **candidata a PK** por su variabilidad frente a la homogeneidad del atributo *fake*. Su ratificación como **PK** depende de validar unicidad y no nulidad para garantizar la **1FN**.

### **2.3.3. Auditoría de tabla fake y validación dimensional (2FN)**

In [220]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_fake_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_fake_limpio, "fake")
df_tabla_fake_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   object
 1   fake    6335 non-null   object
dtypes: object(2)
memory usage: 754.5 KB


**fake - columnas a castear (inicial):** 754.4 KB.

(6335, 2)

**Nota:** Resultado inspección **tabla 'fake'**: el campo **t_id** y **fake** presentan formato **object** requieren **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.3.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.3.3.1.1. Pre-Casting y validación de Dominio**

In [221]:
# Preparación y diagnóstico de tipos de datos
comp_campo_t_id_tabla_fake = df_tabla_fake_limpio["t_id"]                                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_t_id_tabla_fake = comp_campo_t_id_tabla_fake.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_t_id_tabla_fake.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                         # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 't_id' de la tabla 'fake':")                                                                         # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_t_id_tabla_fake.to_string(index=False))                                                                             # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 't_id' de la tabla 'fake':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'t_id'** (actualmente tipado como object) de la '**tabla fake**'. Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [222]:
# Muestra de valores únicos
mask_string_campo_t_id_tabla_fake = df_tabla_fake_limpio["t_id"].apply(lambda x: isinstance(x, str))
string_puros_campo_t_id_tabla_fake = df_tabla_fake_limpio.loc[mask_string_campo_t_id_tabla_fake, "t_id"]
valores_unicos_campo_t_id_tabla_fake = string_puros_campo_t_id_tabla_fake.nunique()

print(f"Valores únicos en el campo 't_id' de la tabla 'fake': {valores_unicos_campo_t_id_tabla_fake}")

Valores únicos en el campo 't_id' de la tabla 'fake': 6335


**Nota:** La cardinalidad de t_id coincide exactamente con el total de registros (6335)

In [223]:
# Evaluación numérica temporal para extraer los límites reales
limite_inferior_campo_t_id_tabla_fake = pd.to_numeric(string_puros_campo_t_id_tabla_fake, errors='coerce').min()
limite_superior_campo_t_id_tabla_fake = pd.to_numeric(string_puros_campo_t_id_tabla_fake, errors='coerce').max()

# Detección de 'strings puros' (textos basura que fallarían en un futuro casting)
nulos_originales_campo_t_id_tabla_fake = string_puros_campo_t_id_tabla_fake.isna().sum()
nulos_evaluados_campo_t_id_tabla_fake = pd.to_numeric(string_puros_campo_t_id_tabla_fake, errors='coerce').isna().sum()
nulos_inducidos_campo_t_id_tabla_fake = nulos_evaluados_campo_t_id_tabla_fake - nulos_originales_campo_t_id_tabla_fake

# Reporte
print("Auditoría de límites del campo 't_id' de la tabla 'fake':")
print("-" * 55)
print(f"Límite inferior numérico (Min): {limite_inferior_campo_t_id_tabla_fake}")
print(f"Límite superior numérico (Max): {limite_superior_campo_t_id_tabla_fake}")

if nulos_inducidos_campo_t_id_tabla_fake > 0:
    print("-" * 55)
    print(f"ALERTA: Se detectan {nulos_inducidos_campo_t_id_tabla_fake} registros de texto no numérico puro.")
else:
    print("-" * 55)
    print("OK: NO detectados STRINGS PUROS que impidan un casting limpio.")

Auditoría de límites del campo 't_id' de la tabla 'fake':
-------------------------------------------------------
Límite inferior numérico (Min): 2
Límite superior numérico (Max): 10557
-------------------------------------------------------
OK: NO detectados STRINGS PUROS que impidan un casting limpio.


**Nota:** El campo '**t_id**' de la '**tabla fake**' **es intrínsecamente numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

##### **2.3.3.1.2. Casting del campo**

In [224]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_fake_limpio["t_id"] = df_tabla_fake_limpio["t_id"].astype("uint16")                                        # Aplicación de downcasting + valores estrictamente positivos
df_tabla_fake_limpio.info(memory_usage='deep')                                                                      # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   fake    6335 non-null   object
dtypes: object(1), uint16(1)
memory usage: 389.9 KB


##### **2.3.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [225]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la 'tabla fake': {len(df_tabla_fake_limpio)}")
print(f"Valores únicos del campo 't_id' en la 'tabla fake': {df_tabla_fake_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la 'tabla fake': {df_tabla_fake_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)


Total de registros de la 'tabla fake': 6335
Valores únicos del campo 't_id' en la 'tabla fake': 6335
Nulos oficiales (isna) del campo 't_id' en la 'tabla fake': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de '**t_id**' de la **tabla** '**fake**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.3.3.2. Atributo 'fake': auditoría, downcasting y unicidad visual.**

##### **2.3.3.2.1. Pre-Casting y validación de Dominio**

In [226]:
# Preparación y diagnóstico de tipos de datos
comp_campo_fake = df_tabla_fake_limpio["fake"]                                                                        # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_fake= comp_campo_fake.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()      # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_fake.columns = ['Tipo de dato', 'Frecuencia absoluta']                                              # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 'fake':")                                                                      # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_fake.to_string(index=False))                                                                  # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 'fake':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


##### **a) Límite del campo**

In [227]:
# Muestra de valores únicos
mask_string_campo_fake = df_tabla_fake_limpio["fake"].apply(lambda x: isinstance(x, str))
string_puros_campo_fake = df_tabla_fake_limpio.loc[mask_string_campo_fake, "fake"]
valores_unicos_campo_fake = string_puros_campo_fake.nunique()

print(f"Valores únicos en el campo 'fake': {valores_unicos_campo_fake}")

Valores únicos en el campo 'fake': 2


##### **Resumen de frecuencia absoluta**

In [228]:
tabla_distribucion_campo_fake = df_tabla_fake_limpio['fake'].value_counts().reset_index()       # Agregación por categorías
tabla_distribucion_campo_fake.columns = ['VALORES ÚNICOS CAMPO FAKE', 'Frecuencia Absoluta']    # Normalización nomenclatura
total = tabla_distribucion_campo_fake['Frecuencia Absoluta'].sum()                              # Cálculo frecuencia absoluta

tabla_distribucion_campo_fake.index = tabla_distribucion_campo_fake.index + 1
tabla_distribucion_campo_fake.loc[""] = ['TOTAL', total]
tabla_distribucion_campo_fake

,VALORES ÚNICOS CAMPO FAKE,Frecuencia Absoluta
1,REAL,3171
2,FAKE,3164
,TOTAL,6335


**Nota:** Se estandariza el atributo **fake al dominio UNISIGNED TINYINT(1)** dado su carácter dicotómico. Se establece una codificación binaria donde **1 denota 'FAKE'** (evento de interés) y **0 'REAL'** (clase de control), asegurando la **paridad de tipos entre Python (uint8) y MySQL**. Esta transformación elimina la redundancia de strings **(3FN)** y optimiza el almacenamiento a 1 byte por registro, facilitando además el cálculo directo de métricas de rendimiento (Precision/Recall).

##### **2.3.3.2.2. Casting del campo**

In [229]:
# Normalización del dominio de 'fake': downcasting a formato numérico y estrictamente positivo (uint8).
df_tabla_fake_limpio['fake'] = df_tabla_fake_limpio['fake'].map({'FAKE': 1, 'REAL': 0})                         # Relación de valores
df_tabla_fake_limpio['fake'] = df_tabla_fake_limpio['fake'].astype('uint8')                                     # Aplicación de downcasting a uint8 para representar 0/1
df_tabla_fake_limpio.info(memory_usage='deep')                                                                  # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   fake    6335 non-null   uint8 
dtypes: uint16(1), uint8(1)
memory usage: 18.7 KB


## **2.4. ⊞ Tabla tiempo**

### **2.4.1. Diagnóstico de integridad y arquitectura del esquema**

In [230]:
# Auditoría técnica de la entidad
tipo_obj_tabla_tiempo = type(tabla_tiempo)                                                                        # Identificación de la clase del objeto
cardinalidad_tabla_tiempo = len(tabla_tiempo)                                                                     # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla tiempo | Clase: {tipo_obj_tabla_tiempo} | Cardinalidad: {cardinalidad_tabla_tiempo}")      # Reporte de consistencia y validación de carga

Entidad: Tabla tiempo | Clase: <class 'dict'> | Cardinalidad: 960


In [231]:
# Inspección del esquema de claves (Keys).
tabla_tiempo.keys()

dict_keys(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157'

In [232]:
# Identificación de dimensiones de clasificación.
tabla_tiempo

{'1': [6220, 2831, 5951],
 '2': [139, 9680, 10054, 1125, 1609, 2767],
 '3': [8146, 1019, 6031, 10027, 8444, 1408, 9658, 6045, 483],
 '4': [6775, 2181, 3788, 60, 3487],
 '5': [3566, 981, 2213, 4201, 2668, 4531],
 '6': [8114, 9227, 2192, 1621, 5172, 7920],
 '7': [3851, 1528, 665, 6832, 2168, 5781, 573],
 '8': [6210, 8056, 525],
 '9': [10133, 7847, 3775, 1550],
 '10': [5575, 5466, 2116, 4895, 1618, 4638, 7066, 9045],
 '11': [7200, 9702, 3409, 2103, 5999, 1373, 3598, 9542, 2043],
 '12': [2076, 6801, 4060, 808, 758],
 '13': [9679, 4774],
 '14': [6437, 7921, 6855, 8302, 6338],
 '15': [90, 8883, 6813, 7160, 2490, 5231, 515],
 '16': [1785, 1644, 5208, 1238, 9058, 7539, 9423],
 '17': [961, 2921, 4875, 3285, 3116, 3352, 5916],
 '18': [8692, 6672, 10233, 9114, 2431, 5862, 1855, 8198, 4064, 3824],
 '19': [9521, 1222, 5213, 1574, 1307],
 '20': [3048, 4308, 6738, 10406, 4902],
 '21': [3529, 628, 807, 6643, 10409, 6440, 873],
 '22': [7713, 6170, 3900, 4824, 878, 122, 6143],
 '23': [6820, 6721, 526, 1

### **2.4.2. Transformación a estructura tabular y normalización del esquema**

In [233]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_tiempo = pd.json_normalize(tabla_tiempo)

#### **2.4.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [234]:
# Diagnóstico de integridad estructural: Detección de esquema horizontal (Wide Format) no relacional.
df_tabla_tiempo

,1,2,3,4,5,6,7,8,9,10,...,951,952,953,954,955,956,957,958,959,960
0,"[6220, 2831, 5951]","[139, 9680, 10054, 1125, 1609, 2767]","[8146, 1019, 6031, 10027, 8444, 1408, 9658, 60...","[6775, 2181, 3788, 60, 3487]","[3566, 981, 2213, 4201, 2668, 4531]","[8114, 9227, 2192, 1621, 5172, 7920]","[3851, 1528, 665, 6832, 2168, 5781, 573]","[6210, 8056, 525]","[10133, 7847, 3775, 1550]","[5575, 5466, 2116, 4895, 1618, 4638, 7066, 9045]",...,"[7680, 205, 10549, 1102, 9356, 1822]","[7049, 3638, 3506, 433, 1720, 1673, 1622]","[8527, 6616, 1285, 1114, 3253, 4082, 5017, 956...","[3117, 9078, 6388, 2261, 8106, 5409]","[3273, 2659, 4472, 1362, 10229]","[7938, 10304, 967, 4958, 529, 5331, 3907, 410,...","[596, 7622, 826, 8770]","[4228, 5300, 5389]","[9625, 7524, 2255, 2815, 4768, 3488, 10519, 71...","[6442, 4253, 4499]"


**Nota:** Identificación de anomalía estructural: presencia de **colecciones anidadas**. El esquema actual colapsa múltiples entidades en un solo registro, **impidiendo la atomicidad del registro** (1FN).

#### **2.4.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [235]:
# Se transforma la granularidad de colecciones (1:N) a instancias atómicas (1:1)
filas_tabla_tiempo = []                                                     # Inicialización del contenedor para la estructura flat (plana).
for tiempo, lista_ids_tabla_tiempo in df_tabla_tiempo.iloc[0].items():      # Iteración sobre el esquema de atributos y sus colecciones asociadas.
   for t_id in lista_ids_tabla_tiempo:                                      # 'Unpacking' de elementos para la generación de registros atómicos.
      filas_tabla_tiempo.append({
            't_id': t_id,                                                   # Mapeo del identificador único (Clave Primaria)
            'tiempo': tiempo                                                # Mapeo del atributo de dimensión (Categoría)
        })
df_tabla_tiempo_limpio = pd.DataFrame(filas_tabla_tiempo)                   # Instanciación del DataFrame final con integridad referencial 1:1.

In [236]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_tiempo_limpio.head(5)

,t_id,tiempo
0,6220,1
1,2831,1
2,5951,1
3,139,2
4,9680,2


**Nota:** Se identifica **t_id** como clave **candidata a PK** por su variabilidad frente a la homogeneidad del atributo *fake*. Su ratificación como **PK** depende de validar unicidad y no nulidad para garantizar la **1FN**.

### **2.4.3. Auditoría de tabla tiempo y validación dimensional (2FN)**

In [237]:
# Inspección de la consistencia de tiempos, recuento de registros y memoria.
df_tabla_tiempo_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_tiempo_limpio, "tiempo")
df_tabla_tiempo_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   int64 
 1   tiempo  6335 non-null   object
dtypes: int64(1), object(1)
memory usage: 420.2 KB


**tiempo - columnas a castear (inicial):** 420.0 KB.

(6335, 2)

**Nota:** Resultado inspección **tabla 'tiempo'**: el campo **t_id** presenta el formato numérico (**int64**). El campo **tiempo** (**object**) requiere **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.4.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.4.3.1.1. Pre-Casting y validación de Dominio**

In [238]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla tiempo' es:", df_tabla_tiempo_limpio['t_id'].min())
print("El valor máximo del campo 't_id' en la 'tabla tiempo' es:", df_tabla_tiempo_limpio['t_id'].max())

El valor mínimo del campo 't_id' en la 'tabla tiempo' es: 2
El valor máximo del campo 't_id' en la 'tabla tiempo' es: 10557


**Nota:** Valores estrictamente positivos y enteros sin riesgo de overflow para la capacidad de uint16.

##### **2.4.3.1.2. Casting del campo**

In [239]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_tiempo_limpio['t_id'] = df_tabla_tiempo_limpio['t_id'].astype('uint16')                                    # Aplicación de downcasting + valores estrictamente positivos
df_tabla_tiempo_limpio.info(memory_usage='deep')                                                                    # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   tiempo  6335 non-null   object
dtypes: object(1), uint16(1)
memory usage: 383.1 KB


##### **2.4.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [240]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la 'tabla tiempo': {len(df_tabla_tiempo_limpio)}")
print(f"Valores únicos del campo 't_id' en la 'tabla tiempo': {df_tabla_tiempo_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la 'tabla tiempo': {df_tabla_tiempo_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)


Total de registros de la 'tabla tiempo': 6335
Valores únicos del campo 't_id' en la 'tabla tiempo': 6335
Nulos oficiales (isna) del campo 't_id' en la 'tabla tiempo': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de '**t_id**' de la **tabla** '**tiempo**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.4.3.2. Atributo 'tiempo': auditoría, downcasting y unicidad visual.**

##### **2.4.3.2.1. Pre-Casting y validación de Dominio**

In [241]:
# Preparación y diagnóstico de tipos de datos
comp_campo_tiempo = df_tabla_tiempo_limpio["tiempo"]                                                                      # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_tiempo = comp_campo_tiempo.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_tiempo.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 'tiempo':")                                                                        # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_tiempo.to_string(index=False))                                                                    # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 'tiempo':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'tiempo'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [242]:
# Muestra de valores únicos
mask_string_campo_tiempo = df_tabla_tiempo_limpio["tiempo"].apply(lambda x: isinstance(x, str))
string_puros_campo_tiempo = df_tabla_tiempo_limpio.loc[mask_string_campo_tiempo, "tiempo"]
valores_unicos_campo_tiempo = string_puros_campo_tiempo.nunique()

print(f"Valores únicos en el campo 'tiempo': {valores_unicos_campo_tiempo}")

Valores únicos en el campo 'tiempo': 958


##### **Resumen de frecuencia absoluta**

In [243]:
tabla_distribucion_campo_tiempo = df_tabla_tiempo_limpio['tiempo'].value_counts().reset_index()       # Agregación por categorías
tabla_distribucion_campo_tiempo.columns = ['VALORES ÚNICOS CAMPO TIEMPO', 'Frecuencia Absoluta']      # Normalización nomenclatura
total_campo_tiempo = tabla_distribucion_campo_tiempo['Frecuencia Absoluta'].sum()                     # Cálculo frecuencia absoluta

tabla_distribucion_campo_tiempo.index = np.arange(1, len(tabla_distribucion_campo_tiempo) + 1)
tabla_distribucion_campo_tiempo.loc[""] = ['TOTAL', total_campo_tiempo]
tabla_distribucion_campo_tiempo

,VALORES ÚNICOS CAMPO TIEMPO,Frecuencia Absoluta
1,956,14
2,767,14
3,561,14
4,522,14
5,875,14
...,...,...
955,392,2
956,506,2
957,799,1
958,344,1


**Nota:** El campo '**tiempo**' es intrínsecamente **numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

In [244]:
# Evaluación numérica temporal para extraer los límites reales
limite_inferior_campo_tiempo = pd.to_numeric(string_puros_campo_tiempo, errors='coerce').min()
limite_superior_campo_tiempo = pd.to_numeric(string_puros_campo_tiempo, errors='coerce').max()

# Detección de 'strings puros' (textos basura que fallarían en un futuro casting)
nulos_originales_campo_tiempo = string_puros_campo_tiempo .isna().sum()
nulos_evaluados_campo_tiempo  = pd.to_numeric(string_puros_campo_tiempo , errors='coerce').isna().sum()
nulos_inducidos_campo_tiempo  = nulos_evaluados_campo_tiempo  - nulos_originales_campo_tiempo

# Reporte
print("Auditoría de límites del campo 'tiempo':")
print("-" * 55)
print(f"Límite inferior numérico (Min): {limite_inferior_campo_tiempo}")
print(f"Límite superior numérico (Max): {limite_superior_campo_tiempo}")

if nulos_inducidos_campo_tiempo  > 0:
    print("-" * 55)
    print(f"ALERTA: Se detectan {nulos_inducidos_campo_tiempo } registros de texto no numérico puro.")
else:
    print("-" * 55)
    print("OK: NO detectados STRINGS PUROS que impidan un casting limpio.")

Auditoría de límites del campo 'tiempo':
-------------------------------------------------------
Límite inferior numérico (Min): 1
Límite superior numérico (Max): 960
-------------------------------------------------------
OK: NO detectados STRINGS PUROS que impidan un casting limpio.


**Nota:** Se confirma que el **campo 'tiempo'** es **estrictamente numérico**, sin nulos, no es PK y que depende de t_id (2FN)

##### **2.4.3.2.2. Casting del campo**

In [245]:
# Normalización del dominio de 'compartir_tiempo': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_tiempo_limpio["tiempo"] = df_tabla_tiempo_limpio["tiempo"].astype("uint16")                                              # Aplicación de downcasting + valores estrictamente positivos
df_tabla_tiempo_limpio.info(memory_usage='deep')                                                                                  # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   tiempo  6335 non-null   uint16
dtypes: uint16(2)
memory usage: 24.9 KB


## **2.5. ⊞ Tabla contenido**

### **2.5.1. Diagnóstico de integridad y arquitectura del esquema**

In [246]:
# Auditoría técnica de la entidad
tipo_obj_tabla_contenido = type(tabla_contenido)                                                                          # Identificación de la clase del objeto
cardinalidad_tabla_contenido = len(tabla_contenido)                                                                       # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla contenido | Clase: {tipo_obj_tabla_contenido} | Cardinalidad: {cardinalidad_tabla_contenido}")     # Reporte de consistencia y validación de carga

Entidad: Tabla contenido | Clase: <class 'dict'> | Cardinalidad: 6335


In [247]:
# Inspección del esquema de claves (Keys).
tabla_contenido.keys()

dict_keys(['8476', '10294', '3608', '10142', '875', '6903', '7341', '95', '4869', '2909', '1357', '988', '7041', '7623', '1571', '4739', '7737', '8716', '3304', '3078', '2517', '10348', '778', '3300', '6155', '636', '755', '626', '691', '5743', '1787', '7808', '6484', '7385', '9016', '9324', '8983', '8965', '5580', '9757', '1967', '431', '5955', '7455', '5224', '7793', '2777', '587', '9403', '5931', '3027', '4377', '5937', '7277', '5521', '3627', '9360', '909', '6646', '1522', '7540', '1834', '1775', '8212', '3363', '1084', '4410', '2669', '3648', '6197', '596', '6061', '1569', '1178', '9545', '1506', '1641', '139', '7744', '8177', '4134', '917', '3854', '7049', '3343', '4348', '3424', '1718', '143', '930', '1661', '7801', '10232', '1360', '3635', '8290', '6227', '3255', '3177', '2832', '4294', '5316', '1524', '2974', '3364', '4933', '5530', '7680', '10490', '9413', '4880', '3512', '8685', '6763', '7730', '2546', '2596', '7362', '7513', '1475', '2232', '6247', '127', '6025', '10122', '

In [248]:
# Identificación de dimensiones de clasificación.
tabla_contenido

{'8476': {'titulo': 'You Can Smell Hillary’s Fear',
  'texto': 'Daniel Greenfield, a Shillman Journalism Fellow at the Freedom Center, is a New York writer focusing on radical Islam. \nIn the final stretch of the election, Hillary Rodham Clinton has gone to war with the FBI. \nThe word “unprecedented” has been thrown around so often this election that it ought to be retired. But it’s still unprecedented for the nominee of a major political party to go war with the FBI. \nBut that’s exactly what Hillary and her people have done. Coma patients just waking up now and watching an hour of CNN from their hospital beds would assume that FBI Director James Comey is Hillary’s opponent in this election. \nThe FBI is under attack by everyone from Obama to CNN. Hillary’s people have circulated a letter attacking Comey. There are currently more media hit pieces lambasting him than targeting Trump. It wouldn’t be too surprising if the Clintons or their allies were to start running attack ads against

### **2.5.2. Transformación a estructura tabular y normalización del esquema**

In [249]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_contenido = pd.json_normalize(tabla_contenido)

#### **2.5.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [250]:
# Diagnóstico de integridad estructural: Detección de esquema horizontal (Wide Format) no relacional.
df_tabla_contenido

,8476.titulo,8476.texto,10294.titulo,10294.texto,3608.titulo,3608.texto,10142.titulo,10142.texto,875.titulo,875.texto,...,4490.titulo,4490.texto,8062.titulo,8062.texto,8622.titulo,8622.texto,4021.titulo,4021.texto,4330.titulo,4330.texto
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,Bernie supporters on Twitter erupt in anger ag...,"- Kaydee King (@KaydeeKing) November 9, 2016 T...",The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,...,State Department says it can't find emails fro...,The State Department told the Republican Natio...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia -President Obama convene...",Jeb Bush Is Suddenly Attacking Trump. Here's W...,Jeb Bush Is Suddenly Attacking Trump. Here's W...


**Nota:** Identificación de anomalía estructural: los **identificadores únicos** se presentan como **cabeceras de columna**, **impidiendo la atomicidad del registro** (1FN).

#### **2.5.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [251]:
# Reorientación de matriz: Transposición del eje de columnas a instancias de filas
df_tabla_contenido_limpio = df_tabla_contenido.T.reset_index()
df_tabla_contenido_limpio.columns = ['key', 'valor']

# Descomposición de atributos jerárquicos (Vectorized String Splitting)
df_tabla_contenido_limpio[['t_id', 'metrica']] = df_tabla_contenido_limpio['key'].str.split('.', n=1, expand=True)

# Pivotaje y restauración de la dimensionalidad para alcanzar la 1FN
df_tabla_contenido_limpio = df_tabla_contenido_limpio.pivot(index='t_id', columns='metrica', values='valor').reset_index()

# Normalización del índice y saneamiento de contenido de cabecera
df_tabla_contenido_limpio.columns.name = None
df_tabla_contenido_limpio = df_tabla_contenido_limpio.reset_index(drop=True)

In [252]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_contenido_limpio.head(5)

,t_id,texto,titulo
0,10,The unexpected death of Justice Antonin Scalia...,Scalia’s death comes just a month before the c...
1,100,(CNN) Ezra Dolezal would love to see his siste...,Rachel Dolezal's brother: She's 'making up mor...
2,10001,"Behind the headlines - conspiracies, cover-ups...",What A Hillary Presidency Would Bring
3,10004,Hillary Endorsed Donald Trump for President Ac...,Hillary Endorsed Donald Trump for President Ac...
4,10006,"RT October 26, 2016 \nAround 300 civilians wer...",US-led coalition killed 300 Syrian civilians i...


### **2.5.3. Auditoría de tabla contenido y validación dimensional (2FN)**

In [253]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_contenido_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_contenido_limpio, "contenido")
df_tabla_contenido_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   object
 1   texto   6335 non-null   object
 2   titulo  6335 non-null   object
dtypes: object(3)
memory usage: 52.8 MB


**contenido - columnas a castear (inicial):** 377.0 KB.

(6335, 3)

**Nota:** Resultado inspección **tabla 'contenido'**: el campo **t_id**, **texto** y **titulo** presentan formato **object** requieren **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.5.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.5.3.1.1. Pre-Casting y validación de Dominio**

In [254]:
# Preparación y diagnóstico de tipos de datos
comp_campo_t_id_tabla_contenido = df_tabla_contenido_limpio["t_id"]                                                                 # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_t_id_tabla_contenido = comp_campo_t_id_tabla_contenido.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()      # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_t_id_tabla_contenido.columns = ['Tipo de dato', 'Frecuencia absoluta']                                              # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 't_id' de la tabla 'contenido':")                                                  # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_t_id_tabla_contenido.to_string(index=False))                                                                  # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 't_id' de la tabla 'contenido':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'t_id'** (actualmente tipado como object) de la '**tabla contenido**'. Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [255]:
# Muestra de valores únicos
mask_string_campo_t_id_tabla_contenido = df_tabla_contenido_limpio["t_id"].apply(lambda x: isinstance(x, str))
string_puros_campo_t_id_tabla_contenido = df_tabla_contenido_limpio.loc[mask_string_campo_t_id_tabla_contenido, "t_id"]
valores_unicos_campo_t_id_tabla_contenido = string_puros_campo_t_id_tabla_contenido.nunique()

print(f"Valores únicos en el campo 't_id' de la tabla 'contenido': {valores_unicos_campo_t_id_tabla_contenido}")

Valores únicos en el campo 't_id' de la tabla 'contenido': 6335


**Nota:** La cardinalidad de t_id coincide exactamente con el total de registros (6335)

In [256]:
# Extracción de límites
string_puros_campo_t_id_tabla_contenido= df_tabla_contenido_limpio["t_id"]

# Evaluación numérica temporal para extraer los límites reales
limite_inferior_campo_t_id_tabla_contenido = pd.to_numeric(string_puros_campo_t_id_tabla_contenido, errors='coerce').min()
limite_superior_campo_t_id_tabla_contenido = pd.to_numeric(string_puros_campo_t_id_tabla_contenido, errors='coerce').max()

# Detección de 'strings puros' (textos basura que fallarían en un futuro casting)
nulos_originales_campo_t_id_tabla_contenido = string_puros_campo_t_id_tabla_contenido.isna().sum()
nulos_evaluados_campo_t_id_tabla_contenido = pd.to_numeric(string_puros_campo_t_id_tabla_contenido, errors='coerce').isna().sum()
nulos_inducidos_campo_t_id_tabla_contenido = nulos_evaluados_campo_t_id_tabla_contenido - nulos_originales_campo_t_id_tabla_contenido

# Reporte
print("Auditoría de límites del campo 't_id' de la tabla 'contenido':")
print("-" * 55)
print(f"Límite inferior numérico (Min): {limite_inferior_campo_t_id_tabla_contenido}")
print(f"Límite superior numérico (Max): {limite_superior_campo_t_id_tabla_contenido}")

if nulos_inducidos_campo_t_id_tabla_contenido > 0:
    print("-" * 55)
    print(f"ALERTA: Se detectan {nulos_inducidos_campo_t_id_tabla_contenido} registros de texto no numérico puro.")
else:
    print("-" * 55)
    print("OK: NO detectados STRINGS PUROS que impidan un casting limpio.")

Auditoría de límites del campo 't_id' de la tabla 'contenido':
-------------------------------------------------------
Límite inferior numérico (Min): 2
Límite superior numérico (Max): 10557
-------------------------------------------------------
OK: NO detectados STRINGS PUROS que impidan un casting limpio.


**Nota:** El campo '**t_id**' de la **tabla 'contenido'**' **es intrínsecamente numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

##### **2.5.3.1.2. Casting del campo**

In [257]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_contenido_limpio["t_id"] = df_tabla_contenido_limpio["t_id"].astype("uint16")                              # Aplicación de downcasting + valores estrictamente positivos
df_tabla_contenido_limpio.info(memory_usage='deep')                                                                 # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   texto   6335 non-null   object
 2   titulo  6335 non-null   object
dtypes: object(2), uint16(1)
memory usage: 52.4 MB


##### **2.5.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [258]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la tabla 'contenido': {len(df_tabla_contenido_limpio)}")
print(f"Valores únicos del campo 't_id' en la tabla 'contenido': {df_tabla_contenido_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la tabla 'contenido': {df_tabla_contenido_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)


Total de registros de la tabla 'contenido': 6335
Valores únicos del campo 't_id' en la tabla 'contenido': 6335
Nulos oficiales (isna) del campo 't_id' en la tabla 'contenido': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de **'t_id' de** la tabla '**contenido**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.5.3.2. Atributo 'texto': auditoría, downcasting y unicidad visual.**

##### **2.5.3.2.1. Pre-Casting y validación de Dominio**

In [259]:
# Preparación y diagnóstico de tipos de datos
comp_campo_texto = df_tabla_contenido_limpio["texto"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_texto = comp_campo_texto.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_texto.columns = ['Tipo de dato', 'Frecuencia absoluta']                                               # Asignación cabeceras a las columnas

print("Tipo objetos Python en campo 'texto':")                                                                          # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_texto.to_string(index=False))                                                                   # Impresión con el formato tabular limpio

Tipo objetos Python en campo 'texto':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'texto'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [260]:
# Muestra de valores únicos
mask_string_campo_texto = df_tabla_contenido_limpio["texto"].apply(lambda x: isinstance(x, str))
string_puros_campo_texto = df_tabla_contenido_limpio.loc[mask_string_campo_texto, "texto"]
valores_unicos_campo_texto = string_puros_campo_texto.nunique()

print(f"Valores únicos en el campo 'texto': {valores_unicos_campo_texto}")

Valores únicos en el campo 'texto': 6060


##### **Resumen de frecuencia absoluta**

In [261]:
tabla_distribucion_campo_texto= df_tabla_contenido_limpio['texto'].value_counts().reset_index()     # Agregación por categorías
tabla_distribucion_campo_texto.columns = ['VALORES ÚNICOS CAMPO TEXTO', 'Frecuencia Absoluta']             # Normalización nomenclatura
total_campo_texto = tabla_distribucion_campo_texto['Frecuencia Absoluta'].sum()                     # Cálculo frecuencia absoluta

tabla_distribucion_campo_texto.index = tabla_distribucion_campo_texto.index + 1
tabla_distribucion_campo_texto.loc[""] = ['TOTAL', total_campo_texto]
tabla_distribucion_campo_texto

,VALORES ÚNICOS CAMPO TEXTO,Frecuencia Absoluta
1,"Killing Obama administration rules, dismantlin...",58
2,,36
3,A verdict in 2017 could have sweeping conseque...,23
4,"The election in 232 photos, 43 numbers and 131...",17
5,Top Dems want White House to call off Part B d...,12
...,...,...
6057,Washington (CNN) President Barack Obama will t...,1
6058,"Sure, We Want An Honest And Trustworthy Presid...",1
6059,President Obama will be conducting visits to t...,1
6060,WHAT WILL Donald Trump do if he loses the elec...,1


**Nota:** Se detectan 6060 valores únicos para el **campo 'texto'**. Al tratarse de descripciones únicas o texto libre, **NO se procede a su división en categorías**.

In [262]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_texto = string_puros_campo_texto.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_texto = longitudes_campo_texto.min()
max_caracteres_campo_texto = longitudes_campo_texto.max()

# Extracción de los registros de los límites
string_mas_corto_campo_texto = str(string_puros_campo_texto.loc[longitudes_campo_texto.idxmin()])
string_mas_largo_campo_texto = str(string_puros_campo_texto.loc[longitudes_campo_texto.idxmax()])

# Límite visual de caracteres para no colapsar el notebook
max_display = 150
trunc_corto_campo_texto = f"{string_mas_corto_campo_texto[:max_display]}..." if len(string_mas_corto_campo_texto) > max_display else string_mas_corto_campo_texto
trunc_largo_campo_texto = f"{string_mas_largo_campo_texto[:max_display]}..." if len(string_mas_largo_campo_texto) > max_display else string_mas_largo_campo_texto

# Reporte
print("Auditoría de límites de longitud del campo 'texto':")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_texto)} caracter. Registro: '{trunc_corto_campo_texto}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_texto)} caracteres. Registro: '{trunc_largo_campo_texto}'")

Auditoría de límites de longitud del campo 'texto':
-----------------------------------------------------------------
Límite inferior (Min): 1 caracter. Registro: ' '
Límite superior (Max): 115372 caracteres. Registro: 'Why Are So Many People Choosing To Leave The United States Permanently?   August 11th, 2013 
Have things gotten so bad that it is time to leave the Un...'


##### **2.5.3.2.1. Casting del campo**

**Nota:** El **campo 'texto' se define como MEDIUMTEXT para la persistencia en MySQL**. A diferencia de los identificadores numéricos (t_id), que se transforman a int en Pandas para optimizar el uso de RAM, los textos se mantienen como object. Dado que Pandas no ofrece una mejora de rendimiento por limitar la longitud de los strings en memoria, esta restricción se delega directamente al motor de la base de datos durante la carga.


#### **2.5.3.3. Atributo 'título': auditoría, downcasting y unicidad visual.**

##### **2.5.3.3.1. Pre-Casting y validación de Dominio**

In [263]:
# Preparación y diagnóstico de tipos de datos
comp_campo_titulo = df_tabla_contenido_limpio["titulo"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_titulo = comp_campo_titulo.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_titulo.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                # Asignación cabeceras a las columnas

print("Tipo objetos Python en campo 'titulo':")                                                                           # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_texto.to_string(index=False))                                                                     # Impresión con el formato tabular limpio

Tipo objetos Python en campo 'titulo':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'titulo'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [264]:
# Muestra de valores únicos
mask_string_campo_titulo = df_tabla_contenido_limpio["titulo"].apply(lambda x: isinstance(x, str))
string_puros_campo_titulo = df_tabla_contenido_limpio.loc[mask_string_campo_titulo, "titulo"]
valores_unicos_campo_titulo = string_puros_campo_titulo.nunique()

print(f"Valores únicos en el campo 'titulo': {valores_unicos_campo_titulo}")

Valores únicos en el campo 'titulo': 6256


##### **Resumen de frecuencia absoluta**

In [265]:
tabla_distribucion_campo_titulo= df_tabla_contenido_limpio['titulo'].value_counts().reset_index()     # Agregación por categorías
tabla_distribucion_campo_titulo.columns = ['VALORES ÚNICOS CAMPO TÍTULO', 'Frecuencia Absoluta']             # Normalización nomenclatura
total_campo_titulo = tabla_distribucion_campo_titulo['Frecuencia Absoluta'].sum()                     # Cálculo frecuencia absoluta

tabla_distribucion_campo_titulo.index = tabla_distribucion_campo_titulo.index + 1
tabla_distribucion_campo_titulo.loc[""] = ['TOTAL', total_campo_titulo]
tabla_distribucion_campo_titulo

,VALORES ÚNICOS CAMPO TÍTULO,Frecuencia Absoluta
1,OnPolitics | 's politics blog,5
2,Hillary’s “Big Tent” is Obama’s “Grand Bargain...,3
3,Michael Moore Owes Me $4.99,3
4,Get Ready For Civil Unrest: Survey Finds That ...,3
5,Tony Blair suggests a second referendum to rev...,2
...,...,...
6253,Security Expert: My Fear is That More Churches...,1
6254,Suspect in 4 gruesome D.C. killings might not ...,1
6255,Church shooting suspect Dylann Roof captured a...,1
6256,Hacker Guccifer 2.0 Issues Dire Warning to Ame...,1


**Nota:** Se detectan 6256 valores únicos para el **campo 'titulo'**. Al tratarse de descripciones únicas o texto libre, **NO se procede a su división en categorías**.

In [266]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_titulo = string_puros_campo_titulo.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_titulo = longitudes_campo_titulo.min()
max_caracteres_campo_titulo = longitudes_campo_titulo.max()

# Extracción de los registros de los límites
string_mas_corto_campo_titulo = str(string_puros_campo_titulo.loc[longitudes_campo_titulo.idxmin()])
string_mas_largo_campo_titulo = str(string_puros_campo_titulo.loc[longitudes_campo_titulo.idxmax()])

# Límite visual de caracteres para no colapsar el notebook
max_display = 150
trunc_corto_campo_titulo = f"{string_mas_corto_campo_titulo[:max_display]}..." if len(string_mas_corto_campo_titulo) > max_display else string_mas_corto_campo_titulo
trunc_largo_campo_titulo = f"{string_mas_largo_campo_titulo[:max_display]}..." if len(string_mas_largo_campo_titulo) > max_display else string_mas_largo_campo_titulo

# Reporte
print("Auditoría de límites de longitud del campo 'titulo':")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_titulo)} caracter. Registro: '{trunc_corto_campo_titulo}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_titulo)} caracteres. Registro: '{trunc_largo_campo_titulo}'")

Auditoría de límites de longitud del campo 'titulo':
-----------------------------------------------------------------
Límite inferior (Min): 1 caracter. Registro: ':'
Límite superior (Max): 289 caracteres. Registro: 'Life: If You Love Enamel Pins, You’ll Love This Etsy Shop, And If You Don’t Care About Enamel Pins, You’ll Explode Over This Picture Of A Bulldog Pupp...'


##### **2.5.3.3.2. Casting del campo**

**Nota:** El **campo 'titulo' se define como VARCHAR(300) para la persistencia en MySQL**. A diferencia de los identificadores numéricos (t_id), que se transforman a int en Pandas para optimizar el uso de RAM, los textos se mantienen como object. Dado que Pandas no ofrece una mejora de rendimiento por limitar la longitud de los strings en memoria, esta restricción se delega directamente al motor de la base de datos durante la carga.


## **2.6. ⊞ Tabla metadatos**

### **2.6.1. Diagnóstico de integridad y arquitectura del esquema**

In [267]:
# Auditoría técnica de la entidad
tipo_obj_tabla_metadatos = type(tabla_metadatos)                                                                          # Identificación de la clase del objeto
cardinalidad_tabla_metadatos = len(tabla_metadatos)                                                                       # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla metadatos | Clase: {tipo_obj_tabla_metadatos} | Cardinalidad: {cardinalidad_tabla_metadatos}")     # Reporte de consistencia y validación de carga

Entidad: Tabla metadatos | Clase: <class 'dict'> | Cardinalidad: 6335


In [268]:
# Inspección del esquema de claves (Keys).
tabla_metadatos.keys()

dict_keys(['8476', '10294', '3608', '10142', '875', '6903', '7341', '95', '4869', '2909', '1357', '988', '7041', '7623', '1571', '4739', '7737', '8716', '3304', '3078', '2517', '10348', '778', '3300', '6155', '636', '755', '626', '691', '5743', '1787', '7808', '6484', '7385', '9016', '9324', '8983', '8965', '5580', '9757', '1967', '431', '5955', '7455', '5224', '7793', '2777', '587', '9403', '5931', '3027', '4377', '5937', '7277', '5521', '3627', '9360', '909', '6646', '1522', '7540', '1834', '1775', '8212', '3363', '1084', '4410', '2669', '3648', '6197', '596', '6061', '1569', '1178', '9545', '1506', '1641', '139', '7744', '8177', '4134', '917', '3854', '7049', '3343', '4348', '3424', '1718', '143', '930', '1661', '7801', '10232', '1360', '3635', '8290', '6227', '3255', '3177', '2832', '4294', '5316', '1524', '2974', '3364', '4933', '5530', '7680', '10490', '9413', '4880', '3512', '8685', '6763', '7730', '2546', '2596', '7362', '7513', '1475', '2232', '6247', '127', '6025', '10122', '

In [269]:
# Identificación de dimensiones de clasificación.
tabla_metadatos

{'8476': {'t_id': 8476,
  'visitas': 389073,
  'compartir': 55938,
  'duracion': 599,
  'favorito': 8008},
 '10294': {'t_id': 10294,
  'visitas': 239160,
  'compartir': 34348,
  'duracion': 216,
  'favorito': 7958},
 '3608': {'t_id': 3608,
  'visitas': 48853,
  'compartir': 43056,
  'duracion': 447,
  'favorito': 8145},
 '10142': {'t_id': 10142,
  'visitas': 710817,
  'compartir': 96927,
  'duracion': 184,
  'favorito': 1102},
 '875': {'t_id': 875,
  'visitas': 997280,
  'compartir': 74788,
  'duracion': 175,
  'favorito': 3299},
 '6903': {'t_id': 6903,
  'visitas': 732305,
  'compartir': 74878,
  'duracion': -17,
  'favorito': 3672},
 '7341': {'t_id': 7341,
  'visitas': 417548,
  'compartir': 88078,
  'duracion': 350,
  'favorito': 1117},
 '95': {'t_id': 95,
  'visitas': 485990,
  'compartir': 19274,
  'duracion': 56,
  'favorito': 3452},
 '4869': {'t_id': 4869,
  'visitas': 494533,
  'compartir': 90885,
  'duracion': 262,
  'favorito': 6342},
 '2909': {'t_id': 2909,
  'visitas': 7007

### **2.6.2. Transformación a estructura tabular y normalización del esquema**

In [270]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_metadatos = pd.json_normalize(tabla_metadatos)

#### **2.6.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [271]:
# Diagnóstico de integridad estructural: Detección de esquema horizontal (Wide Format) no relacional.
df_tabla_metadatos

,8476.t_id,8476.visitas,8476.compartir,8476.duracion,8476.favorito,10294.t_id,10294.visitas,10294.compartir,10294.duracion,10294.favorito,...,4021.t_id,4021.visitas,4021.compartir,4021.duracion,4021.favorito,4330.t_id,4330.visitas,4330.compartir,4330.duracion,4330.favorito
0,8476,389073,55938,599,8008,10294,239160,34348,216,7958,...,4021,722283,14882,428,5709,4330,593554,68616,373,568


**Nota:** Identificación de anomalía estructural: los **identificadores únicos** se presentan como **cabeceras de columna**, **impidiendo la atomicidad del registro** (1FN).

#### **2.6.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [272]:
# Reorientación de matriz: Transposición del eje de columnas a instancias de filas
df_stack = df_tabla_metadatos.T.reset_index()                                                       # Transposición del eje de columnas a instancias de filas
df_stack.columns = ['columna', 'valor']                                                             # Definición de esquema temporal para el parsing de atributos

# Descomposición de atributos jerárquicos (Vectorized String Splitting)
df_stack[['grupo_id', 'metrica']] = df_stack['columna'].str.split('.', expand=True)                 # Segregación del identificador de grupo (id) y la métrica

# Pivotaje y restauración de la dimensionalidad
df_tabla_metadatos_limpio = df_stack.pivot(index='grupo_id', columns='metrica', values='valor')     # Reconstrucción del DataFrame para alcanzar la atomicidad por registro (1FN)

# Reordenación heurística y jerarquización de columnas
cols_ordenadas = ['t_id'] + [c for c in df_tabla_metadatos_limpio.columns if c != 't_id']           # Normalización del orden de los atributos
df_tabla_metadatos_limpio = df_tabla_metadatos_limpio[cols_ordenadas]

# Normalización del índice y saneamiento de metadatos de cabecera
df_tabla_metadatos_limpio.columns.name = None                                                       # Eliminación de la etiqueta de nivel superior (metrica)
df_tabla_metadatos_limpio = df_tabla_metadatos_limpio.reset_index(drop=True)                        # Reseteo del índice para continuidad secuencial de los registros


In [273]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_metadatos_limpio.head(5)

,t_id,compartir,duracion,favorito,visitas
0,10,59275,558,5035,24881
1,100,60949,279,9823,737334
2,10001,10094,432,5226,563200
3,10004,30962,171,5078,883120
4,10006,51808,166,6424,870165


### **2.6.3. Auditoría de tabla metadatos y validación dimensional (2FN)**

In [274]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_metadatos_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_metadatos_limpio, "metadatos")
df_tabla_metadatos_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   object
 1   compartir  6320 non-null   object
 2   duracion   6335 non-null   object
 3   favorito   6223 non-null   object
 4   visitas    6333 non-null   object
dtypes: object(5)
memory usage: 1.1 MB


**metadatos - columnas a castear (inicial):** 1114.8 KB.

(6335, 5)

**Nota:** Resultado inspección **tabla 'metadatos'**: el campo **t_id**, **compartir**, **duracion**, **favorito** y **visitas** presentan formato **object** requieren **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.6.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.6.3.1.1. Pre-Casting y validación de Dominio**

In [275]:
# Preparación y diagnóstico de tipos de datos
comp_campo_t_id_tabla_metadatos = df_tabla_metadatos_limpio["t_id"]                                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_t_id_tabla_metadatos = comp_campo_t_id_tabla_metadatos.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_t_id_tabla_metadatos.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                              # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 't_id' de la tabla 'metadatos':")                                                                              # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_t_id_tabla_metadatos.to_string(index=False))                                                                                  # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 't_id' de la tabla 'metadatos':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         int                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'t_id'** (actualmente tipado como object) de la '**tabla metadatos**'. Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [276]:
# Muestra de valores únicos
mask_string_campo_t_id_tabla_metadatos = df_tabla_metadatos_limpio["t_id"].apply(lambda x: isinstance(x, int))
string_puros_campo_t_id_tabla_metadatos = df_tabla_metadatos_limpio.loc[mask_string_campo_t_id_tabla_metadatos, "t_id"]
valores_unicos_campo_t_id_tabla_metadatos = string_puros_campo_t_id_tabla_metadatos.nunique()

print(f"Valores únicos en el campo 't_id' de la tabla 'metadatos': {valores_unicos_campo_t_id_tabla_metadatos}")

Valores únicos en el campo 't_id' de la tabla 'metadatos': 6335


**Nota:** La cardinalidad de t_id coincide exactamente con el total de registros (6335)

In [277]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla metadatos' es:", df_tabla_metadatos_limpio['t_id'].min())
print("El valor máximo del campo 't_id' en la 'tabla metadatos' es:", df_tabla_metadatos_limpio['t_id'].max())

El valor mínimo del campo 't_id' en la 'tabla metadatos' es: 2
El valor máximo del campo 't_id' en la 'tabla metadatos' es: 10557


**Nota:** El campo '**t_id**' de la **tabla 'metadatos'**' **es intrínsecamente numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

##### **2.6.3.1.2. Casting del campo**

In [278]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_metadatos_limpio["t_id"] = df_tabla_metadatos_limpio["t_id"].astype("uint16")                              # Aplicación de downcasting + valores estrictamente positivos
df_tabla_metadatos_limpio.info(memory_usage='deep')                                                                 # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   uint16
 1   compartir  6320 non-null   object
 2   duracion   6335 non-null   object
 3   favorito   6223 non-null   object
 4   visitas    6333 non-null   object
dtypes: object(4), uint16(1)
memory usage: 904.6 KB


##### **2.6.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [279]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la tabla 'metadatos': {len(df_tabla_metadatos_limpio)}")
print(f"Valores únicos del campo 't_id' en la tabla 'metadatos': {df_tabla_metadatos_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la tabla 'metadatos': {df_tabla_metadatos_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)


Total de registros de la tabla 'metadatos': 6335
Valores únicos del campo 't_id' en la tabla 'metadatos': 6335
Nulos oficiales (isna) del campo 't_id' en la tabla 'metadatos': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de **'t_id' de** la tabla '**metadatos**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.6.3.2. Atributo 'compartir': auditoría, downcasting y unicidad visual.**

##### **2.6.3.2.1. Pre-Casting y validación de Dominio**

In [280]:
# Preparación y diagnóstico de tipos de datos
comp_campo_compartir = df_tabla_metadatos_limpio["compartir"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_compartir = comp_campo_compartir.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_compartir.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                   # Asignación cabeceras a las columnas

print("Tipo objetos Python en 'compartir':")                                                                                    # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_compartir.to_string(index=False))                                                                       # Impresión con el formato tabular limpio

Tipo objetos Python en 'compartir':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         int                 6296
    NoneType                   15
       float                   12
         str                   12


**Nota:** Se confirma la **presencia de tipos mixtos** en el campo **'compartir'** (actualmente tipado como object). Se procederá con la **auditoría de cada tipo mixto** para validar el rango y ejecutar un downcasting numérico seguro que optimice la memoria.

##### **a) Auditoría tipos mixtos**

In [281]:
# Auditoría de precisión de string () para el tipo de dato mixto del campo 'compartir'
tabla_textos_ocultos_campo_compartir = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, str)), "compartir"].value_counts().reset_index()      # Filtrado y contabilización valores
tabla_textos_ocultos_campo_compartir.columns = ['Valor string', 'Frecuencia absoluta']                                                                                                          # Asignación cabeceras a las columnas

print("Tipo string en 'compartir':")                                                                                                                                                            # Cabecera para el reporte
print("-" * 45)
print(tabla_textos_ocultos_campo_compartir.to_string(index=False))                                                                                                                              # Impresión con el formato tabular limpio

Tipo string en 'compartir':
---------------------------------------------
Valor string  Frecuencia absoluta
        null                   12


In [282]:
# Extracción de los identificadores (t_id) de estos registros para su posterior análisis en el EDA
mask_null_string_campo_compartir = df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, str) and x.lower() == "null")
ids_compartir_null_campo_compartir = df_tabla_metadatos_limpio.loc[mask_null_string_campo_compartir, "t_id"].tolist()

print("Lista t_ids con valor 'null' (string) en el campo 'compartit':")
print(ids_compartir_null_campo_compartir)

Lista t_ids con valor 'null' (string) en el campo 'compartit':
[1770, 2543, 3489, 3990, 4096, 7059, 7836, 7869, 8673, 887, 8991, 9085]


**Nota:** Se aísla el **texto "null"** en el **campo 'compartir'** para corregir la anomalía de ingesta y mantener la trazabilidad del linaje de datos.
Esta **conversión a nulos** de sistema es un paso indispensable para habilitar el futuro downcasting numérico seguro.
Además, **extraer los t_id afectados** permitirá **evaluar si existen patrones de error sistemáticos durante el EDA**.

In [283]:
# Auditoría de precisión de float () para el tipo de dato mixto del campo 'compartir'
tabla_floats_compartir_campo_compartir = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, float)), "compartir"].value_counts().reset_index()      # Filtrado y contabilización de valores
tabla_floats_compartir_campo_compartir.columns = ['Valor float', 'Frecuencia absoluta']                                                                                                             # Asignación cabeceras a las columnas

print("Tipo float en el campo 'compartir':")                                                                                                                                                        # Cabecera para el reporte
print("-" * 45)
print(tabla_floats_compartir_campo_compartir.to_string(index=False))                                                                                                                                # Impresión con el formato tabular limpio

Tipo float en el campo 'compartir':
---------------------------------------------
 Valor float  Frecuencia absoluta
        72.5                   12


In [284]:
total_73_campo_compartir = (df_tabla_metadatos_limpio["compartir"] == 73).sum()
print(f"Total de registros con valor 73 en el campo 'compartir': {total_73_campo_compartir}")

Total de registros con valor 73 en el campo 'compartir': 0


**Nota:** **Estandarización de anomalías decimales**. Para viabilizar el tipado entero de la variable discreta 'compartir' sin distorsionar las métricas de nulidad, se aplica una imputación controlada de los **12 registros anómalos (72.5) asignándoles un valor inédito en la distribución (73)**. Esta estrategia consolida la coherencia estructural del dato y **garantiza una trazabilidad exacta para su posterior evaluación durante el EDA**.

In [285]:
# Auditoría de valores negativos aislando de forma segura los tipos numéricos
mask_negativos_campo_compartir = df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, (int, float)) and x < 0)

# Filtrado y contabilización
tabla_negativos_campo_compartir = df_tabla_metadatos_limpio.loc[mask_negativos_campo_compartir, "compartir"].value_counts().reset_index()

tabla_negativos_campo_compartir.columns = ['Valor negativo', 'Frecuencia absoluta']

print("Valores negativos en el campo 'compartir':")
print("-" * 45)
print(tabla_negativos_campo_compartir.to_string(index=False))                                                                                          # Impresión con el formato tabular limpio

Valores negativos en el campo 'compartir':
---------------------------------------------
 Valor negativo  Frecuencia absoluta
             -1                   10


**Nota:** **Preservación analítica y regularización diferida**. Se opta por **mantener intactos los valores (-1) en el campo 'compartir' para asegurar la trazabilidad del error de origen durante el EDA**. La corrección geométrica de estas métricas (**positivación**) **se delega** estratégicamente a la **fase de persistencia en MySQL**, permitiendo la exploración estadística de las anomalías sin comprometer la integridad del modelo de datos final.

##### **b) Límite del campo**

In [286]:
# Auditoría de valores enteros aislando de forma segura los tipos numéricos para el correcto casting
mask_numeros_campo_compartir = df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, (int, float)))
numeros_puros_campo_compartir = df_tabla_metadatos_limpio.loc[mask_numeros_campo_compartir, "compartir"]

# Extracción de límites
limite_inferior_campo_compartir = numeros_puros_campo_compartir.min()
limite_superior_campo_compartir = numeros_puros_campo_compartir.max()

# Reporte de límites para el downcasting
print("Límites del campo 'compartir' (solo datos numéricos):")
print("-" * 55)
print(f"Límite inferior (Min): {limite_inferior_campo_compartir}")
print(f"Límite superior (Max): {limite_superior_campo_compartir}")

Límites del campo 'compartir' (solo datos numéricos):
-------------------------------------------------------
Límite inferior (Min): -1
Límite superior (Max): 99951


**Nota:** Valores sin riesgo de overflow para la capacidad de int32. Conversión forzosa de los falsos nulos (strings "null") a nulos matemáticos reales antes de aplicar la transformación de tipo.

##### **2.6.3.2.2. Casting del campo**

In [287]:
# Forzado null (string) del campo 'compartir' a nulos matemáticos
mask_null_string_campo_compartir = df_tabla_metadatos_limpio["compartir"].apply(lambda x: isinstance(x, str) and x.lower() == "null")
df_tabla_metadatos_limpio.loc[mask_null_string_campo_compartir, "compartir"] = np.nan

# Reporte de control
print(f"Se han forzado {mask_null_string_campo_compartir.sum()} strings 'null' a nulos nativos (np.nan).")

Se han forzado 12 strings 'null' a nulos nativos (np.nan).


In [288]:
# Redondeo de floats al valor entero inmediatamente más próximo.
mask_72_5_campo_compartir = df_tabla_metadatos_limpio["compartir"] == 72.5
df_tabla_metadatos_limpio.loc[mask_72_5_campo_compartir ,"compartir"] = 73

# Reporte de control
total_73_campo_compartir = (df_tabla_metadatos_limpio["compartir"] == 73).sum()
print(f"Total de registros con valor 73: {total_73_campo_compartir}")

Total de registros con valor 73: 12


In [289]:
# Normalización del dominio de 'compartir': downcasting a formato numérico y estrictamente positivo (Int16).

df_tabla_metadatos_limpio["compartir"] = df_tabla_metadatos_limpio["compartir"].astype("Int32")                         # Aplicación de downcasting + valores estrictamente positivos
df_tabla_metadatos_limpio.info(memory_usage='deep')                                                                     # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   uint16
 1   compartir  6308 non-null   Int32 
 2   duracion   6335 non-null   object
 3   favorito   6223 non-null   object
 4   visitas    6333 non-null   object
dtypes: Int32(1), object(3), uint16(1)
memory usage: 712.8 KB


#### **2.6.3.3. Atributo 'duración': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.6.3.3.1. Pre-Casting y validación de Dominio**

In [290]:
# Preparación y diagnóstico de tipos de datos
comp_campo_duracion = df_tabla_metadatos_limpio["duracion"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_duracion = comp_campo_duracion.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_duracion.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                  # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 'duracion':")                                                                          # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_duracion.to_string(index=False))                                                                      # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 'duracion':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         int                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'duracion'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [291]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla metadatos' es:", df_tabla_metadatos_limpio['duracion'].min())
print("El valor máximo del campo 't_id' en la 'tabla metadatos' es:", df_tabla_metadatos_limpio['duracion'].max())

El valor mínimo del campo 't_id' en la 'tabla metadatos' es: -512
El valor máximo del campo 't_id' en la 'tabla metadatos' es: 961


**Nota:** Casting campo 'duración': Valores sin riesgo de overflow para la capacidad de int16.

##### **Resumen de frecuencia absoluta**

In [292]:
# Auditoría de valores negativos aislando de forma segura los tipos numéricos
mask_negativos_campo_duracion = df_tabla_metadatos_limpio["duracion"].apply(lambda x: isinstance(x, (int, float)) and x < 0)

# Filtrado y contabilización
tabla_negativos_campo_duracion = df_tabla_metadatos_limpio.loc[mask_negativos_campo_duracion, "duracion"].value_counts().reset_index()

tabla_negativos_campo_duracion.columns = ['VALORES ÚNICOS Y NEGATIVOS CAMPO DURACIÓN', 'Frecuencia absoluta']

print("Valores negativos en el campo 'duracion':")
print("-" * 45)
print(tabla_negativos_campo_duracion.to_string(index=False))                                                                                          # Impresión con el formato tabular limpio

Valores negativos en el campo 'duracion':
---------------------------------------------
 VALORES ÚNICOS Y NEGATIVOS CAMPO DURACIÓN  Frecuencia absoluta
                                       -33                   10
                                       -28                    9
                                       -96                    8
                                        -4                    8
                                       -35                    8
                                       -20                    8
                                        -2                    7
                                       -32                    7
                                        -8                    7
                                       -52                    7
                                        -5                    7
                                       -24                    7
                                       -18                    7
                

**Nota:** Se **conservan los valores negativos** bajo el principio de no intervención para asegurar la **trazabilidad** del error durante el **EDA**. La **positivación** necesaria para los cálculos estadísticos de 'duración' se delegará a la **fase de persistencia en MySQL**.

##### **2.6.3.3.2. Casting del campo**

In [293]:
# Normalización del dominio de 'duración': downcasting a formato numérico y estrictamente positivo (int16).

df_tabla_metadatos_limpio["duracion"] = df_tabla_metadatos_limpio["duracion"].astype("int16")                           # Aplicación de downcasting + valores estrictamente positivos
df_tabla_metadatos_limpio.info(memory_usage='deep')                                                                     # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   uint16
 1   compartir  6308 non-null   Int32 
 2   duracion   6335 non-null   int16 
 3   favorito   6223 non-null   object
 4   visitas    6333 non-null   object
dtypes: Int32(1), int16(1), object(2), uint16(1)
memory usage: 502.5 KB


#### **2.6.3.4. Atributo 'favorito': auditoría, downcasting y unicidad visual.**

##### **2.6.3.4.1. Pre-Casting y validación de Dominio**

In [294]:
# Preparación y diagnóstico de tipos de datos
comp_campo_favorito = df_tabla_metadatos_limpio["favorito"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_favorito = comp_campo_favorito.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_favorito.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                  # Asignación cabeceras a las columnas

print("Tipo objetos Python en campo 'favorito':")                                                                             # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_favorito.to_string(index=False))                                                                      # Impresión con el formato tabular limpio

Tipo objetos Python en campo 'favorito':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         int                 5975
       float                  126
         str                  122
    NoneType                  112


**Nota:** Se confirma la **presencia de tipos mixtos** en el campo **'favorito'** (actualmente tipado como object). Se procederá con la **auditoría de cada tipo mixto** para validar el rango y ejecutar un downcasting numérico seguro que optimice la memoria.

##### **a) Auditoría tipos mixtos**

In [295]:
# Auditoría de precisión de string () para el tipo de dato mixto del campo 'compartir'
tabla_textos_ocultos_campo_favorito = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["favorito"].apply(lambda x: isinstance(x, str)), "favorito"].value_counts().reset_index()      # Filtrado y contabilización valores
tabla_textos_ocultos_campo_favorito.columns = ['Valor string', 'Frecuencia absoluta']                                                                                                          # Asignación cabeceras a las columnas

print("Tipo string en campo 'favorito':")                                                                                                                                                            # Cabecera para el reporte
print("-" * 45)
print(tabla_textos_ocultos_campo_favorito.to_string(index=False))                                                                                                                              # Impresión con el formato tabular limpio

Tipo string en campo 'favorito':
---------------------------------------------
Valor string  Frecuencia absoluta
        favx                  122


**Nota:** Se identifican **122 registros** anómalos tipo string ("favx") en el campo '**favorito**'. Se **reemplazarán por el valor centinela -1** para habilitar el casting numérico general. Esta estrategia sanea la estructura del dato **garantizando la trazabilidad** de la anomalía **para su evaluación en el EDA**.

In [296]:
# Auditoría de precisión de float () para el tipo de dato mixto del campo 'favorito'
tabla_floats_campo_favorito = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["favorito"].apply(lambda x: isinstance(x, float)), "favorito"].value_counts().reset_index()      # Filtrado y contabilización de valores
tabla_floats_campo_favorito.columns = ['Valor float', 'Frecuencia absoluta']                                                                                                           # Asignación cabeceras a las columnas

print("Tipo float en el campo 'favorito':")                                                                                                                                            # Cabecera para el reporte
print("-" * 45)
print(tabla_floats_campo_favorito.to_string(index=False))                                                                                                                              # Impresión con el formato tabular limpio

Tipo float en el campo 'favorito':
---------------------------------------------
 Valor float  Frecuencia absoluta
        0.73                  126


In [297]:
total_1_campo_favorito = (df_tabla_metadatos_limpio["favorito"] == 1).sum()
print(f"Total de registros con valor 1 en el campo 'favorito': {total_1_campo_favorito}")

Total de registros con valor 1 en el campo 'favorito': 1


In [298]:
# Extracción de los identificadores (t_id) de estos registros para su posterior análisis en el EDA
mask_1_campo_favorito = df_tabla_metadatos_limpio["favorito"].isin([1, "1"])      # Uso de 'isin' para no alterar la naturaleza del campo
ids_campo_favorito_1 = df_tabla_metadatos_limpio.loc[mask_1_campo_favorito, "t_id"].tolist()

print("Lista de t_ids con valor 1 en el campo 'favorito' antes del redondeo de los registros float al mismo valor:")
print(ids_campo_favorito_1)

Lista de t_ids con valor 1 en el campo 'favorito' antes del redondeo de los registros float al mismo valor:
[305]


**Nota:** **Normalización discreta**: Para habilitar el casting del campo favorito a tipo entero sin alterar las métricas de nulos, se han imputado los **126 registros fraccionarios (0.73) al valor 1**. Esta **imputación controlada** se alinea con el único **registro preexistente de valor 1 (cuyo t_id ha sido aislado)**, garantizando la coherencia matemática de la variable y una **trazabilidad estricta para el EDA**.

In [299]:
# Auditoría de valores negativos aislando de forma segura los tipos numéricos
mask_negativos_campo_favorito = df_tabla_metadatos_limpio["favorito"].apply(lambda x: isinstance(x, (int, float)) and x < 0)

# Filtrado y contabilización
tabla_negativos_campo_favorito = df_tabla_metadatos_limpio.loc[mask_negativos_campo_favorito, "favorito"].value_counts().reset_index()

tabla_negativos_campo_favorito.columns = ['Valor negativo', 'Frecuencia absoluta']

print("Valores negativos en el campo 'favorito':")
print("-" * 45)
print(tabla_negativos_campo_favorito.to_string(index=False))                                                                                          # Impresión con el formato tabular limpio

Valores negativos en el campo 'favorito':
---------------------------------------------
Empty DataFrame
Columns: [Valor negativo, Frecuencia absoluta]
Index: []


**Nota**: **No hay presencia de valores negativos**. Aunque el campo no percibirá un cast 'unsigned' debido al uso del valor centinela -1 para los registros anómalos 'favx'. No influye en el 'overflow' del campo.

##### **Límite del campo**

In [300]:
# Auditoría de valores enteros aislando de forma segura los tipos numéricos para el correcto casting
mask_numeros_campo_favorito = df_tabla_metadatos_limpio["favorito"].apply(lambda x: isinstance(x, (int, float)))
numeros_puros_campo_favorito = df_tabla_metadatos_limpio.loc[mask_numeros_campo_favorito, "favorito"]

# Extracción de límites
limite_inferior_campo_favorito = numeros_puros_campo_favorito.min()
limite_superior_campo_favorito = numeros_puros_campo_favorito.max()

# Reporte de límites para el downcasting
print("Límites del campo 'favorito' (solo datos numéricos):")
print("-" * 55)
print(f"Límite inferior (Min): {limite_inferior_campo_favorito}")
print(f"Límite superior (Max): {limite_superior_campo_favorito}")

Límites del campo 'favorito' (solo datos numéricos):
-------------------------------------------------------
Límite inferior (Min): 0.73
Límite superior (Max): 9998


**Nota:** Valores sin riesgo de overflow para la capacidad de int16. Conversión forzosa de los valors strings "favx" a valores centinela numérico (-1) y redondeo del float 0,73 a valor entero 1 antes de aplicar la transformación del campo favorito.

##### **2.6.3.4.2. Casting del campo**

In [301]:
# 1. Creación de la máscara vectorizada (blindada contra espacios y mayúsculas)
mask_favx_campo_favorito = df_tabla_metadatos_limpio["favorito"].astype(str).str.strip().str.lower() == "favx"

# 2. Conteo previo
total_reemplazos_favx_campo_favorito = mask_favx_campo_favorito.sum()

# 3. Forzado al valor centinela numérico (mejor -1 que '-1' para un futuro casting)
df_tabla_metadatos_limpio.loc[mask_favx_campo_favorito, "favorito"] = -1

# 4. Reporte de control
print(f"Se han forzado {total_reemplazos_favx_campo_favorito} strings 'favx' a valor centinela (-1).")

Se han forzado 122 strings 'favx' a valor centinela (-1).


In [302]:
# Redondeo de floats al valor entero inmediatamente más próximo.
mask_0_73_campo_favorito = df_tabla_metadatos_limpio["favorito"] == 0.73
df_tabla_metadatos_limpio.loc[mask_0_73_campo_favorito ,"favorito"] = 1

# Reporte de control
total_1_campo_favorito = (df_tabla_metadatos_limpio["favorito"] == 1).sum()
print(f"Total de registros con valor 1: {total_1_campo_favorito}")

Total de registros con valor 1: 127


In [303]:
# Normalización del dominio de 'favorito': downcasting a formato numérico y estrictamente positivo (Int16).

df_tabla_metadatos_limpio["favorito"] = df_tabla_metadatos_limpio["favorito"].astype("Int16")                           # Aplicación de downcasting + valores estrictamente positivos
df_tabla_metadatos_limpio.info(memory_usage='deep')                                                                     # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   uint16
 1   compartir  6308 non-null   Int32 
 2   duracion   6335 non-null   int16 
 3   favorito   6223 non-null   Int16 
 4   visitas    6333 non-null   object
dtypes: Int16(1), Int32(1), int16(1), object(1), uint16(1)
memory usage: 297.2 KB


#### **2.6.3.5. Atributo 'visitas': auditoría, downcasting y unicidad visual.**

##### **2.6.3.5.1. Pre-Casting y validación de Dominio**

In [304]:
# Preparación y diagnóstico de tipos de datos
comp_campo_visitas = df_tabla_metadatos_limpio["visitas"]                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_visitas = comp_campo_visitas.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_visitas.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                 # Asignación cabeceras a las columnas

print("Tipo objetos Python en campo 'visitas':")                                                                            # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_visitas.to_string(index=False))                                                                     # Impresión con el formato tabular limpio

Tipo objetos Python en campo 'visitas':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         int                 6327
         str                    5
    NoneType                    2
       float                    1


**Nota:** Se confirma la **presencia de tipos mixtos** en el campo **'visitas'** (actualmente tipado como object). Se procederá con la **auditoría de cada tipo mixto** para validar el rango y ejecutar un downcasting numérico seguro que optimice la memoria.

##### **a) Auditoría tipos mixtos**

In [305]:
# Auditoría de precisión de string () para el tipo de dato mixto del campo 'compartir'
tabla_textos_ocultos_campo_visitas = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["visitas"].apply(lambda x: isinstance(x, str)), "visitas"].value_counts().reset_index()      # Filtrado y contabilización valores
tabla_textos_ocultos_campo_visitas.columns = ['Valor string', 'Frecuencia absoluta']                                                                                                          # Asignación cabeceras a las columnas

print("Tipo string en campo 'visitas':")                                                                                                                                                            # Cabecera para el reporte
print("-" * 45)
print(tabla_textos_ocultos_campo_visitas.to_string(index=False))                                                                                                                              # Impresión con el formato tabular limpio

Tipo string en campo 'visitas':
---------------------------------------------
Valor string  Frecuencia absoluta
         err                    5


**Nota:** Se identifican **5 registros** anómalos tipo string ("err") en el campo '**visitas**'. Se **reemplazarán por valor nulo** para habilitar el casting numérico general. Los t_id de estos seran asilados. Esta **imputación controlada** garantiza la coherencia matemática de la variable y una **trazabilidad estricta para el EDA**.

In [306]:
# Extracción de los identificadores (t_id) de estos registros para su posterior análisis en el EDA
mask_err_campo_visitas = df_tabla_metadatos_limpio["visitas"].isin([1, "err"])                                        # Uso de 'isin' para no alterar la naturaleza del campo
ids_campo_visitas_err = df_tabla_metadatos_limpio.loc[mask_err_campo_visitas, "t_id"].tolist()

print("Lista de t_ids con valor string 'err' en el campo 'visitas' antes del reemplazo de los registros a NaN:")
print(ids_campo_visitas_err)

Lista de t_ids con valor string 'err' en el campo 'visitas' antes del reemplazo de los registros a NaN:
[10340, 4141, 4845, 6902, 808]


In [307]:
# Auditoría de precisión de float () para el tipo de dato mixto del campo 'visitas'
tabla_floats_campo_visitas = df_tabla_metadatos_limpio.loc[df_tabla_metadatos_limpio["visitas"].apply(lambda x: isinstance(x, float)), "visitas"].value_counts().reset_index()      # Filtrado y contabilización de valores
tabla_floats_campo_visitas.columns = ['Valor float', 'Frecuencia absoluta']                                                                                                         # Asignación cabeceras a las columnas

print("Tipo float en el campo 'visitas':")                                                                                                                                          # Cabecera para el reporte
print("-" * 45)
print(tabla_floats_campo_visitas.to_string(index=False))                                                                                                                            # Impresión con el formato tabular limpio

Tipo float en el campo 'visitas':
---------------------------------------------
 Valor float  Frecuencia absoluta
         3.1                    1


In [308]:
total_3_campo_visitas = (df_tabla_metadatos_limpio["visitas"] == 3).sum()
print(f"Total de registros con valor 3 en el campo 'visitas': {total_3_campo_visitas}")

Total de registros con valor 3 en el campo 'visitas': 0


**Nota:** **Normalización discreta**: Para habilitar el casting del campo favorito a tipo entero sin alterar las métricas de nulos, se ha imputado el **registros fraccionarios (3.1) al valor 3**. Esta **imputación controlada** garantiza la coherencia matemática de la variable y una **trazabilidad estricta para el EDA**.

In [309]:
# Auditoría de valores negativos aislando de forma segura los tipos numéricos
mask_negativos_campo_visitas = df_tabla_metadatos_limpio["visitas"].apply(lambda x: isinstance(x, (int, float)) and x < 0)

# Filtrado y contabilización
tabla_negativos_campo_visitas = df_tabla_metadatos_limpio.loc[mask_negativos_campo_visitas, "visitas"].value_counts().reset_index()

tabla_negativos_campo_visitas.columns = ['Valor negativo', 'Frecuencia absoluta']

print("Valores negativos en el campo 'visitas':")
print("-" * 45)
print(tabla_negativos_campo_visitas.to_string(index=False))                                                                             # Impresión con el formato tabular limpio

Valores negativos en el campo 'visitas':
---------------------------------------------
 Valor negativo  Frecuencia absoluta
             -7                    1


**Nota:** Se **conserva el valore negativo** bajo el principio de no intervención para asegurar la **trazabilidad** del error durante el **EDA**. La **positivación** necesaria para los cálculos estadísticos de **'visitas'** se delegará a la **fase de persistencia en MySQL**.

##### **b) Límite del campo**

In [310]:
# Auditoría de valores enteros aislando de forma segura los tipos numéricos para el correcto casting
mask_numeros_campo_visitas = df_tabla_metadatos_limpio["visitas"].apply(lambda x: isinstance(x, (int, float)))
numeros_puros_campo_visitas = df_tabla_metadatos_limpio.loc[mask_numeros_campo_visitas, "visitas"]

# Extracción de límites
limite_inferior_campo_visitas = numeros_puros_campo_visitas.min()
limite_superior_campo_visitas = numeros_puros_campo_visitas.max()

# Reporte de límites para el downcasting
print("Límites del campo 'visitas' (solo datos numéricos):")
print("-" * 55)
print(f"Límite inferior (Min): {limite_inferior_campo_visitas}")
print(f"Límite superior (Max): {limite_superior_campo_visitas}")

Límites del campo 'visitas' (solo datos numéricos):
-------------------------------------------------------
Límite inferior (Min): -7
Límite superior (Max): 999864


**Nota:** Valores sin riesgo de overflow para la capacidad de int32. Conversión forzosa de los valors strings "err" a NaN, redondeo del float 3,1 a valor entero 3 antes de aplicar la transformación del campo 'visitas'.

##### **2.6.3.5.2. Casting del campo**

In [311]:
# Creación de la máscara vectorizada (blindada contra espacios y mayúsculas)
mask_err_campo_visitas = df_tabla_metadatos_limpio["visitas"].astype(str).str.strip().str.lower() == "err"

# Conteo previo
total_reemplazos_err_campo_visitas = mask_err_campo_visitas.sum()

# Forzado al valor centinela numérico (mejor -1 que '-1' para un futuro casting)
df_tabla_metadatos_limpio.loc[mask_err_campo_visitas, "visitas"] = np.nan

# Reporte de control
print(f"Se han forzado {total_reemplazos_err_campo_visitas} strings 'err' a valor nulo np.nan.")

Se han forzado 5 strings 'err' a valor nulo np.nan.


In [312]:
# Redondeo de floats al valor entero inmediatamente más próximo.
mask_3_1_campo_visitas = df_tabla_metadatos_limpio["visitas"] == 3.1
df_tabla_metadatos_limpio.loc[mask_3_1_campo_visitas ,"visitas"] = 3

# Reporte de control
total_3_campo_visitas = (df_tabla_metadatos_limpio["visitas"] == 3).sum()
print(f"Total de registros con valor 3: {total_3_campo_visitas}")

Total de registros con valor 3: 1


In [313]:
# Normalización del dominio de 'visitas': downcasting a formato numérico (Int32).

df_tabla_metadatos_limpio["visitas"] = df_tabla_metadatos_limpio["visitas"].astype("Int32")     # Aplicación de downcasting
df_tabla_metadatos_limpio.info()                                                                # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   t_id       6335 non-null   uint16
 1   compartir  6308 non-null   Int32 
 2   duracion   6335 non-null   int16 
 3   favorito   6223 non-null   Int16 
 4   visitas    6328 non-null   Int32 
dtypes: Int16(1), Int32(2), int16(1), uint16(1)
memory usage: 105.3 KB


## **2.7. ⊞ Tabla pais_idioma**

### **2.7.1. Diagnóstico de integridad y arquitectura del esquema**

In [314]:
# Auditoría técnica de la entidad
tipo_obj_tabla_pais_idioma = type(tabla_pais_idioma)                                                                            # Identificación de la clase del objeto
cardinalidad_tabla_pais_idioma = len(tabla_pais_idioma)                                                                         # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla pais_idioma | Clase: {tipo_obj_tabla_pais_idioma} | Cardinalidad: {cardinalidad_tabla_pais_idioma}")     # Reporte de consistencia y validación de carga

Entidad: Tabla pais_idioma | Clase: <class 'list'> | Cardinalidad: 6335


In [315]:
# Identificación de dimensiones de clasificación.
tabla_pais_idioma[:5]

['8476---United States###English',
 '10294---Bangladesh###English',
 '3608---Tanzania###English',
 '10142---Thailand###English',
 '875---Kenya###English']

### **2.7.2. Transformación a estructura tabular y normalización del esquema**

In [316]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_pais_idioma = pd.json_normalize(tabla_pais_idioma)

#### **2.7.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [317]:
# Diagnóstico de integridad estructural: detección de la estructura sin proyección de columnas de la lista anidada.
df_tabla_pais_idioma

""
0
1
2
3
4
...
6330
6331
6332
6333


**Nota:** Identificación de anomalía estructural: **todos los valores** se registran en **1:0**, sin relación, **impidiendo la atomicidad del registro** (1FN).

#### **2.7.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [318]:
# Extracción y segmentación de datos mediante expresiones regulares
df_tabla_pais_idioma_limpio = pd.Series(tabla_pais_idioma).str.split(r'---|###', expand=True)     # Transformación de la lista original en una estructura tabular usando 2 delimitadores (--- o ###)

# Definición del esquema y nombrado de columnas
df_tabla_pais_idioma_limpio.columns = ["t_id", "pais", "idioma"]                                  # Asignación de nombres descriptivos para facilitar la manipulación de los atributos



In [319]:
# Validación preliminar de la consistencia estructural del registro limpio
df_tabla_pais_idioma_limpio.head(5)

,t_id,pais,idioma
0,8476,United States,English
1,10294,Bangladesh,English
2,3608,Tanzania,English
3,10142,Thailand,English
4,875,Kenya,English


### **2.7.3. Auditoría de tabla país_idioma y validación dimensional (2FN)**

In [320]:
# Inspección de la consistencia de tipos, recuento de registros y memoria.
df_tabla_pais_idioma_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_pais_idioma_limpio, "pais_idioma")
df_tabla_pais_idioma_limpio.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   object
 1   pais    6335 non-null   object
 2   idioma  6335 non-null   object
dtypes: object(3)
memory usage: 1.1 MB


**pais_idioma - columnas a castear (inicial):** 377.0 KB.

(6335, 3)

**Nota:** Resultado inspección **tabla 'pais_idioms'**: el campo **t_id**, **pais** e **idioma** presentan formato **object** requieren **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.7.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.7.3.1.1. Pre-Casting y validación de Dominio**

In [321]:
# Preparación y diagnóstico de tipos de datos
comp_campo_t_id_tabla_pais_idioma = df_tabla_pais_idioma_limpio["t_id"]                                                                                   # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_t_id_tabla_pais_idioma = comp_campo_t_id_tabla_pais_idioma.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_t_id_tabla_pais_idioma.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                                # Asignación cabeceras a las columnas

print("Tipo objetos Python en el campo 't_id' de la tabla 'pais_idioma':")                                                                                # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_t_id_tabla_pais_idioma.to_string(index=False))                                                                                    # Impresión con el formato tabular limpio

Tipo objetos Python en el campo 't_id' de la tabla 'pais_idioma':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'t_id'** (actualmente tipado como object) de la '**tabla pais_idioma**'. Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [322]:
# Muestra de valores únicos
mask_string_campo_t_id_tabla_pais_idioma = df_tabla_pais_idioma_limpio["t_id"].apply(lambda x: isinstance(x, str))
string_puros_campo_t_id_tabla_pais_idioma = df_tabla_pais_idioma_limpio.loc[mask_string_campo_t_id_tabla_pais_idioma, "t_id"]
valores_unicos_campo_t_id_tabla_pais_idioma = string_puros_campo_t_id_tabla_pais_idioma.nunique()

print(f"Valores únicos en el campo 't_id' de la tabla 'pais_idioma': {valores_unicos_campo_t_id_tabla_pais_idioma}")

Valores únicos en el campo 't_id' de la tabla 'pais_idioma': 6335


**Nota:** La cardinalidad de t_id coincide exactamente con el total de registros (6335)

In [323]:
# Extracción de límites
string_puros_campo_t_id_tabla_pais_idioma = df_tabla_pais_idioma_limpio["t_id"]

# Evaluación numérica temporal para extraer los límites reales
limite_inferior_campo_t_id_tabla_pais_idioma = pd.to_numeric(string_puros_campo_t_id_tabla_pais_idioma, errors='coerce').min()
limite_superior_campo_t_id_tabla_pais_idioma = pd.to_numeric(string_puros_campo_t_id_tabla_pais_idioma, errors='coerce').max()

# Detección de 'strings puros' (textos basura que fallarían en un futuro casting)
nulos_originales_campo_t_id_tabla_pais_idioma = string_puros_campo_t_id_tabla_pais_idioma.isna().sum()
nulos_evaluados_campo_t_id_tabla_pais_idioma = pd.to_numeric(string_puros_campo_t_id_tabla_pais_idioma, errors='coerce').isna().sum()
nulos_inducidos_campo_t_id_tabla_pais_idioma = nulos_evaluados_campo_t_id_tabla_pais_idioma - nulos_originales_campo_t_id_tabla_pais_idioma

# Reporte
print("Auditoría de límites del campo 't_id' de la tabla 'pais_idioma':")
print("-" * 55)
print(f"Límite inferior numérico (Min): {limite_inferior_campo_t_id_tabla_pais_idioma}")
print(f"Límite superior numérico (Max): {limite_superior_campo_t_id_tabla_pais_idioma}")

if nulos_inducidos_campo_t_id_tabla_pais_idioma > 0:
    print("-" * 55)
    print(f"ALERTA: Se detectan {nulos_inducidos_campo_t_id_tabla_pais_idioma} registros de texto no numérico puro.")
else:
    print("-" * 55)
    print("OK: NO detectados STRINGS PUROS que impidan un casting limpio.")

Auditoría de límites del campo 't_id' de la tabla 'pais_idioma':
-------------------------------------------------------
Límite inferior numérico (Min): 2
Límite superior numérico (Max): 10557
-------------------------------------------------------
OK: NO detectados STRINGS PUROS que impidan un casting limpio.


**Nota:** El campo '**t_id**' de la '**tabla pais_idioma**' **es intrínsecamente numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

##### **2.7.3.1.2. Casting del campo**

In [324]:
# Normalización del dominio de 'pais_idioma': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_pais_idioma_limpio["t_id"] = df_tabla_pais_idioma_limpio["t_id"].astype("uint16")                                  # Aplicación de downcasting + valores estrictamente positivos
df_tabla_pais_idioma_limpio.info(memory_usage='deep')                                                                       # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   t_id    6335 non-null   uint16
 1   pais    6335 non-null   object
 2   idioma  6335 non-null   object
dtypes: object(2), uint16(1)
memory usage: 807.0 KB


##### **2.7.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [325]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la tabla 'pais_idioma': {len(df_tabla_pais_idioma_limpio)}")
print(f"Valores únicos del campo 't_id' en la tabla 'pais_idioma': {df_tabla_pais_idioma_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la tabla 'pais_idioma': {df_tabla_pais_idioma_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)

Total de registros de la tabla 'pais_idioma': 6335
Valores únicos del campo 't_id' en la tabla 'pais_idioma': 6335
Nulos oficiales (isna) del campo 't_id' en la tabla 'pais_idioma': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de **'t_id' de** la tabla '**pais_idioma**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.7.3.2. Atributo 'país': auditoría, downcasting y unicidad visual.**

##### **2.7.3.2.1. Pre-Casting y validación de Dominio**

In [326]:
# Preparación y diagnóstico de tipos de datos
comp_campo_pais = df_tabla_pais_idioma_limpio["pais"]                                                                 # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_pais = comp_campo_pais.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_pais.columns = ['Tipo de dato', 'Frecuencia absoluta']                                              # Asignación cabeceras a las columnas

print("Tipo objetos Python en 'pais':")                                                                               # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_pais.to_string(index=False))                                                                  # Impresión con el formato tabular limpio

Tipo objetos Python en 'pais':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'pais'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [327]:
# Muestra de valores únicos
mask_string_campo_pais = df_tabla_pais_idioma_limpio["pais"].apply(lambda x: isinstance(x, str))
string_puros_campo_pais = df_tabla_pais_idioma_limpio.loc[mask_string_campo_pais, "pais"]
valores_unicos_campo_pais = string_puros_campo_pais.nunique()

print(f"Valores únicos en el campo 'pais': {valores_unicos_campo_pais}")

Valores únicos en el campo 'pais': 35


##### **Resumen de frecuencia absoluta**

In [328]:
tabla_distribucion_campo_pais= df_tabla_pais_idioma_limpio['pais'].value_counts().reset_index()     # Agregación por categorías
tabla_distribucion_campo_pais.columns = ['VALORES ÚNICOS CAMPO PAÍS', 'Frecuencia Absoluta']     # Normalización nomenclatura
total_campo_pais = tabla_distribucion_campo_pais['Frecuencia Absoluta'].sum()                       # Cálculo frecuencia absoluta

tabla_distribucion_campo_pais.index = tabla_distribucion_campo_pais.index + 1
tabla_distribucion_campo_pais.loc[""] = ['TOTAL', total_campo_pais]
tabla_distribucion_campo_pais

,VALORES ÚNICOS CAMPO PAÍS,Frecuencia Absoluta
1,Myanmar,216
2,United Kingdom,201
3,Kenya,195
4,India,193
5,United States,191
6,Brazil,191
7,Bangladesh,191
8,Ukraine,190
9,Philippines,189
10,Sudan,189


**Nota:** Para el campo '**país**' Se detectan **35 categorías únicas**. Para evitar redundancia de datos este **texto se extraerá a una tabla catálogo (dimensión)**, dejando únicamente su ID (3FN).

In [329]:
# Cálculo de las longitudes de todos los strings
longitudes_campo_pais = string_puros_campo_pais.str.len()

# Extracción de los límites de caracteres
min_caracteres_campo_pais = longitudes_campo_pais.min()
max_caracteres_campo_pais = longitudes_campo_pais.max()

# Extracción de las registros de los limites
string_mas_corto_campo_pais = string_puros_campo_pais.loc[longitudes_campo_pais.idxmin()]
string_mas_largo_campo_pais = string_puros_campo_pais.loc[longitudes_campo_pais.idxmax()]

# Reporte
print("Auditoría de límites de longitud del campo 'pais' (Categorías):")
print("-" * 65)
print(f"Límite inferior (Min): {int(min_caracteres_campo_pais)} caracteres. Registro '{string_mas_corto_campo_pais}'")
print(f"Límite superior (Max): {int(max_caracteres_campo_pais)} caracteres. Registro '{string_mas_largo_campo_pais}'")

Auditoría de límites de longitud del campo 'pais' (Categorías):
-----------------------------------------------------------------
Límite inferior (Min): 4 caracteres. Registro 'Iran'
Límite superior (Max): 14 caracteres. Registro 'United Kingdom'


##### **2.7.3.2.2. Casting del campo**

**Nota:** Se determina que la optimización del tipo de dato para el atributo categórico **'pais' (definido como VARCHAR(30))** se ejecutará en la fase de persistencia en **MySQL**. Debido a que en el entorno de desarrollo (Pandas) los strings se gestionan como objetos de memoria, la segregación y el casting final se realizarán mediante la creación de la **Tabla Maestra [PAIS_ID vs. PAIS]**. Esta estrategia asegura la integridad referencial y el cumplimiento estricto de la Tercera Forma Normal **(3FN)**, optimizando la eficiencia de almacenamiento y evitando la redundancia de datos.


#### **2.7.3.3. Atributo 'idioma': auditoría.**

##### **2.7.3.3.1. Pre-Casting y validación de Dominio**

In [330]:
# Preparación y diagnóstico de tipos de datos
comp_campo_idioma = df_tabla_pais_idioma_limpio["idioma"]                                                                 # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_idioma = comp_campo_idioma.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_idioma.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                # Asignación cabeceras a las columnas

print("Tipo objetos Python en 'idioma':")                                                                                 # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_idioma.to_string(index=False))                                                                    # Impresión con el formato tabular limpio

Tipo objetos Python en 'idioma':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'idioma'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [331]:
# Muestra de valores únicos
mask_string_campo_idioma = df_tabla_pais_idioma_limpio["idioma"].apply(lambda x: isinstance(x, str))
string_puros_campo_idioma = df_tabla_pais_idioma_limpio.loc[mask_string_campo_idioma, "idioma"]
valores_unicos_campo_idioma = string_puros_campo_idioma.nunique()

print(f"Valores únicos en el campo 'idioma': {valores_unicos_campo_idioma}")

Valores únicos en el campo 'idioma': 1


##### **Resumen de frecuencia absoluta**

In [332]:
tabla_distribucion_campo_idioma= df_tabla_pais_idioma_limpio['idioma'].value_counts().reset_index()     # Agregación por categorías
tabla_distribucion_campo_idioma.columns = ['VALORES ÚNICOS CAMPO IDIOMA', 'Frecuencia Absoluta']               # Normalización nomenclatura
total_campo_idioma = tabla_distribucion_campo_idioma['Frecuencia Absoluta'].sum()                       # Cálculo frecuencia absoluta

tabla_distribucion_campo_idioma.index = tabla_distribucion_campo_idioma.index + 1
tabla_distribucion_campo_idioma.loc[""] = ['TOTAL', total_campo_idioma]
tabla_distribucion_campo_idioma

,VALORES ÚNICOS CAMPO IDIOMA,Frecuencia Absoluta
1,English,6335
,TOTAL,6335


##### **2.7.3.3.2. Casting del campo**

**Nota:** El **campo idioma** es una **constante del dataset** indicativo de que el alcanze del proyecto se limita a noticias en inglés no presentándose como un atributo intrínseco de cada registro. Por ello, **se descarta su inclusión para asegurar la Tercera Forma Normal (3FN)**, eliminando dependencias transitivas y redundancias que no aportan valor predictivo al esquema.


## **2.8. ⊞ Tabla compartir tiempo**

### **2.8.1. Diagnóstico de integridad y arquitectura del esquema**

In [333]:
# Auditoría técnica de la entidad
tipo_obj_tabla_compartir_tiempo = type(tabla_compartir_tiempo)                                                                            # Identificación de la clase del objeto
cardinalidad_compartir_tiempo = len(tabla_compartir_tiempo)                                                                               # Evaluación del volumen de registros para el pipeline de 1FN

print(f"Entidad: Tabla compartir tiempo | Clase: {tipo_obj_tabla_compartir_tiempo} | Cardinalidad: {cardinalidad_compartir_tiempo}")      # Reporte de consistencia y validación de carga

Entidad: Tabla compartir tiempo | Clase: <class 'dict'> | Cardinalidad: 960


In [334]:
# Inspección del esquema de claves (Keys).
tabla_compartir_tiempo.keys()

dict_keys(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', '157'

In [335]:
# Identificación de dimensiones de clasificación.
tabla_compartir_tiempo

{'1': [4013],
 '2': [8521, 10531, 7867, 726, 8545, 7266, 3787],
 '3': [6784, 8925, 1577, 10126, 7632, 7323],
 '4': [8983, 9771, 2797, 4740, 4528],
 '5': [3398, 6865, 4759, 7091, 7249, 10244],
 '6': [5206, 379, 4183, 6419, 2158, 6734, 349],
 '7': [4489, 8699, 6881, 8508, 10299, 10222],
 '8': [1700, 10179, 2106, 4947, 54, 3407, 6894, 8090, 5904],
 '9': [248, 2836, 4926],
 '10': [347, 1179, 5690],
 '11': [7713,
  8206,
  5663,
  9629,
  3194,
  3285,
  10339,
  5287,
  4575,
  3068,
  7618,
  7334],
 '12': [1146, 2616, 4452, 1125, 9246, 765, 9672],
 '13': [4239, 5207, 4887, 3714, 5897],
 '14': [6884],
 '15': [7006, 781, 3418, 6255, 2872, 7915],
 '16': [6690, 2827, 9513],
 '17': [162, 9930, 6830, 2801],
 '18': [5602, 9698, 8490],
 '19': [7236, 1133],
 '20': [9696, 7007, 4571, 4156, 6228],
 '21': [6128, 5876, 281, 6417, 3503, 2667, 3283],
 '22': [9231,
  8762,
  5939,
  2259,
  6400,
  5519,
  8222,
  9207,
  6878,
  6971,
  2379,
  6751],
 '23': [2392, 6561, 1168, 4326, 2440, 1091, 6737, 5

### **2.8.2. Transformación a estructura tabular y normalización del esquema**

In [336]:
# Aplanado (Flattening) y normalización del esquema jerárquico.
df_tabla_compartir_tiempo = pd.json_normalize(tabla_compartir_tiempo)

#### **2.8.2.1. Diagnóstico de anomalías estructurales: Auditoría de granularidad, anidamiento y dispersión dimensional**

In [337]:
# Diagnóstico de integridad estructural: Detección de esquema horizontal (Wide Format) no relacional.
df_tabla_compartir_tiempo

,1,2,3,4,5,6,7,8,9,10,...,951,952,953,954,955,956,957,958,959,960
0,[4013],"[8521, 10531, 7867, 726, 8545, 7266, 3787]","[6784, 8925, 1577, 10126, 7632, 7323]","[8983, 9771, 2797, 4740, 4528]","[3398, 6865, 4759, 7091, 7249, 10244]","[5206, 379, 4183, 6419, 2158, 6734, 349]","[4489, 8699, 6881, 8508, 10299, 10222]","[1700, 10179, 2106, 4947, 54, 3407, 6894, 8090...","[248, 2836, 4926]","[347, 1179, 5690]",...,"[1951, 5623, 269, 9278, 3353, 2059, 8480, 3245...","[1641, 5727, 9157, 2160, 4612, 5191]","[9917, 4542, 9664, 2412, 92, 5675, 1040, 2322]","[9764, 8632, 2221, 8804, 468, 1408, 7056, 9721]","[5760, 9093, 4047, 465, 5905, 1148, 1459, 6282...","[1115, 7438, 7794, 3869, 4591, 1335, 9837, 761...","[667, 7789, 5484, 368, 1931, 2793]","[6918, 4561, 6729, 7161, 9103, 9955, 2958, 191...","[10372, 10441, 9153, 10268, 1061, 7594, 3166]","[6646, 4134, 9114, 4439, 3258, 206, 1350, 7122]"


**Nota:** Identificación de anomalía estructural: presencia de **colecciones anidadas**. El esquema actual colapsa múltiples entidades en un solo registro, **impidiendo la atomicidad del registro** (1FN).

#### **2.8.2.2. Reestructuración lógica y desanidado manual de registros (1FN)**

In [338]:
# Se transforma la granularidad de colecciones (1:N) a instancias atómicas (1:1)
filas_tabla_compartir_tiempo = []                                                                         # Inicialización del contenedor para la estructura flat (plana).
for compartir_tiempo, lista_ids_tabla_compartir_tiempo in df_tabla_compartir_tiempo.iloc[0].items():      # Iteración sobre el esquema de atributos y sus colecciones asociadas.
   for t_id in lista_ids_tabla_compartir_tiempo:                                                          # 'Unpacking' de elementos para la generación de registros atómicos.
      filas_tabla_compartir_tiempo.append({
            't_id': t_id,                                                                                 # Mapeo del identificador único (Clave Primaria)
            'compartir_tiempo': compartir_tiempo                                                          # Mapeo del atributo de dimensión (Categoría)
        })
df_tabla_compartir_tiempo_limpio = pd.DataFrame(filas_tabla_compartir_tiempo)                             # Instanciación del DataFrame final con integridad referencial 1:1.

In [339]:
# Validación de la nueva estructura tabular y consistencia de registros.
df_tabla_compartir_tiempo_limpio.head(5)

,t_id,compartir_tiempo
0,4013,1
1,8521,2
2,10531,2
3,7867,2
4,726,2


### **2.8.3. Auditoría de tabla compartir tiempo y validación dimensional (2FN)**

In [340]:
# Inspección de la consistencia de tiempos, recuento de registros y memoria.
df_tabla_compartir_tiempo_limpio.info(memory_usage='deep')
memoria_inicial(df_tabla_compartir_tiempo_limpio, "compartir_tiempo")
df_tabla_compartir_tiempo_limpio.shape


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   t_id              6335 non-null   int64 
 1   compartir_tiempo  6335 non-null   object
dtypes: int64(1), object(1)
memory usage: 420.2 KB


**compartir_tiempo - columnas a castear (inicial):** 420.0 KB.

(6335, 2)

**Nota:** Resultado inspección **tabla 'compartir tiempo'**: el campo **t_id** presenta el formato numérico (**int64**). El campo **compartir_tiempo** (**object**) requiere **inspección adicional para aislar y tratar posibles tipos mixtos** antes del casting.

#### **2.8.3.1. Atributo 't_id': auditoría, downcasting y unicidad visual. Candidato a PK.**

##### **2.8.3.1.1. Pre-Casting y validación de Dominio**

In [341]:
# Verificación previa del rango para evitar desbordamiento (overflow)
print("El valor mínimo del campo 't_id' en la 'tabla compartir tiempo' es:", df_tabla_compartir_tiempo_limpio['t_id'].min())
print("El valor máximo del campo 't_id' en la 'tabla compartir tiempo' es:", df_tabla_compartir_tiempo_limpio['t_id'].max())

El valor mínimo del campo 't_id' en la 'tabla compartir tiempo' es: 2
El valor máximo del campo 't_id' en la 'tabla compartir tiempo' es: 10557


**Nota:** Valores estrictamente positivos y enteros sin riesgo de overflow para la capacidad de uint16.

##### **2.8.3.1.2. Casting del campo**

In [342]:
# Normalización del dominio de 't_id': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_compartir_tiempo_limpio['t_id'] = df_tabla_compartir_tiempo_limpio['t_id'].astype('uint16')                # Aplicación de downcasting + valores estrictamente positivos
df_tabla_compartir_tiempo_limpio.info(memory_usage='deep')                                                          # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   t_id              6335 non-null   uint16
 1   compartir_tiempo  6335 non-null   object
dtypes: object(1), uint16(1)
memory usage: 383.0 KB


##### **2.8.3.1.3. Verificación técnica de la Primary Key (t_id)**

In [343]:
# Confirmar ausencia de nulos lógicos y duplicados
print(f"Total de registros de la tabla 'compartir_tiempo': {len(df_tabla_compartir_tiempo_limpio)}")
print(f"Valores únicos del campo 't_id' en la tabla 'compartir_tiempo': {df_tabla_compartir_tiempo_limpio['t_id'].nunique()}")                # Verificacion de unicidad (1FN)
print(f"Nulos oficiales (isna) del campo 't_id' en la tabla 'compartir_tiempo': {df_tabla_compartir_tiempo_limpio['t_id'].isna().sum()}")     # Verificacion de nulos (Descarte de:("" o " "), "null", "None" o "N/A" y 0's)


Total de registros de la tabla 'compartir_tiempo': 6335
Valores únicos del campo 't_id' en la tabla 'compartir_tiempo': 6335
Nulos oficiales (isna) del campo 't_id' en la tabla 'compartir_tiempo': 0


**Nota:** **Conclusión técnica para el proyecto:** Al confirmar la ausencia de nulos lógicos y coincidir el total de registros con la cardinalidad de '**t_id**' de la **tabla** '**compartir_tiempo**', **se valida** su rol como Clave Primaria **(PK)** bajo la 1ª Forma Normal (1FN).

#### **2.8.3.2. Atributo 'compartir tiempo': auditoría, downcasting y unicidad visual.**

##### **2.8.3.2.1. Pre-Casting y validación de Dominio**

In [344]:
# Preparación y diagnóstico de tipos de datos
comp_campo_compartir_tiempo = df_tabla_compartir_tiempo_limpio["compartir_tiempo"]                                                      # Extracción del feature (columna) para su análisis individual
tabla_tipos_campo_compartir_tiempo  = comp_campo_compartir_tiempo.map(lambda x: type(x).__name__).value_counts(dropna=False).reset_index()     # Conteo de tipo de dato y dictament del DataFrame
tabla_tipos_campo_compartir_tiempo.columns = ['Tipo de dato', 'Frecuencia absoluta']                                                    # Asignación cabeceras a las columnas

print("Tipo objetos Python en 'compartir_tiempo':")                                                                               # Cabecera para el reporte de tipos de objetos
print("-" * 45)
print(tabla_tipos_campo_compartir_tiempo.to_string(index=False))                                                                        # Impresión con el formato tabular limpio

Tipo objetos Python en 'compartir_tiempo':
---------------------------------------------
Tipo de dato  Frecuencia absoluta
         str                 6335


**Nota:** Se confirma la **ausencia de tipos mixtos** en el campo **'compartir_tiempo'** (actualmente tipado como object). Se procederá con la **auditoría de los límites** para validar el rango y ejecutar un downcasting seguro que optimice la memoria.

##### **a) Límite del campo**

In [345]:
# Muestra de valores únicos
mask_string_campo_compartir_tiempo  = df_tabla_compartir_tiempo_limpio["compartir_tiempo"].apply(lambda x: isinstance(x, str))
string_puros_campo_compartir_tiempo  = df_tabla_compartir_tiempo_limpio.loc[mask_string_campo_compartir_tiempo, "compartir_tiempo"]
valores_unicos_campo_compartir_tiempo = string_puros_campo_compartir_tiempo.nunique()

print(f"Valores únicos en el campo 'compartir_tiempo': {valores_unicos_campo_compartir_tiempo}")

Valores únicos en el campo 'compartir_tiempo': 959


##### **Resumen de frecuencia absoluta**

In [346]:
tabla_distribucion_campo_compartir_tiempo = df_tabla_compartir_tiempo_limpio['compartir_tiempo'].value_counts().reset_index()     # Agregación por categorías
tabla_distribucion_campo_compartir_tiempo.columns = ['VALORES ÚNICOS CAMPO COMPARTIR_TIEMPO', 'Frecuencia Absoluta']                             # Normalización nomenclatura
total_campo_compartir_tiempo = tabla_distribucion_campo_compartir_tiempo['Frecuencia Absoluta'].sum()                                   # Cálculo frecuencia absoluta

tabla_distribucion_campo_compartir_tiempo.index = np.arange(1, len(tabla_distribucion_campo_compartir_tiempo) + 1)
tabla_distribucion_campo_compartir_tiempo.loc[""] = ['TOTAL', total_campo_compartir_tiempo]
tabla_distribucion_campo_compartir_tiempo

,VALORES ÚNICOS CAMPO COMPARTIR_TIEMPO,Frecuencia Absoluta
1,948,19
2,229,16
3,462,16
4,195,15
5,719,15
...,...,...
956,695,1
957,498,1
958,891,1
959,1,1


**Nota:** El campo '**compartir_tiempo**' es intrínsecamente **numérico**, pero sus valores están capturados como texto (object/str). Se aplicará un casting para restaurar su naturaleza matemática y habilitar su correcto análisis.

In [347]:
# Evaluación numérica temporal para extraer los límites reales
limite_inferior_campo_compartir_tiempo = pd.to_numeric(string_puros_campo_compartir_tiempo, errors='coerce').min()
limite_superior_campo_compartir_tiempo = pd.to_numeric(string_puros_campo_compartir_tiempo, errors='coerce').max()

# Detección de 'strings puros' (textos basura que fallarían en un futuro casting)
nulos_originales_campo_compartir_tiempo = string_puros_campo_compartir_tiempo.isna().sum()
nulos_evaluados_campo_compartir_tiempo = pd.to_numeric(string_puros_campo_compartir_tiempo, errors='coerce').isna().sum()
nulos_inducidos_campo_compartir_tiempo = nulos_evaluados_campo_compartir_tiempo - nulos_originales_campo_compartir_tiempo

# Reporte
print("Auditoría de límites del campo 'compartir_tiempo':")
print("-" * 55)
print(f"Límite inferior numérico (Min): {limite_inferior_campo_compartir_tiempo}")
print(f"Límite superior numérico (Max): {limite_superior_campo_compartir_tiempo}")

if nulos_inducidos_campo_compartir_tiempo > 0:
    print("-" * 55)
    print(f"ALERTA: Se detectan {nulos_inducidos_campo_compartir_tiempo} registros de texto no numérico puro.")
else:
    print("-" * 55)
    print("OK: NO detectados STRINGS PUROS que impidan un casting limpio.")

Auditoría de límites del campo 'compartir_tiempo':
-------------------------------------------------------
Límite inferior numérico (Min): 1
Límite superior numérico (Max): 960
-------------------------------------------------------
OK: NO detectados STRINGS PUROS que impidan un casting limpio.


**Nota:** Se confirma que el **campo 'compartir_tiempo' es estrictamente numérico, sin nulos, no es PK y que depende de t_id (2FN)**

##### **2.8.3.2.2. Casting del campo**

In [348]:
# Normalización del dominio de 'compartir_tiempo': downcasting a formato numérico y estrictamente positivo (uint16).
df_tabla_compartir_tiempo_limpio["compartir_tiempo"] = df_tabla_compartir_tiempo_limpio["compartir_tiempo"].astype("uint16")            # Aplicación de downcasting + valores estrictamente positivos
df_tabla_compartir_tiempo_limpio.info(memory_usage='deep')                                                                              # Validación del cambio y del ahorro de memoria

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6335 entries, 0 to 6334
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   t_id              6335 non-null   uint16
 1   compartir_tiempo  6335 non-null   uint16
dtypes: uint16(2)
memory usage: 24.9 KB


## **2.9. Redución del uso de memoria del dataset después del downcasting en variables numéricas.**

In [349]:
tablas = {
    "info": df_tabla_info_limpio, "tipo": df_tabla_tipo_limpio,
    "fake": df_tabla_fake_limpio, "tiempo": df_tabla_tiempo_limpio,
    "contenido": df_tabla_contenido_limpio, "metadatos": df_tabla_metadatos_limpio,
    "pais_idioma": df_tabla_pais_idioma_limpio, "compartir_tiempo": df_tabla_compartir_tiempo_limpio,
}
antes = sum(_mem_ini.values())
ahora = sum(_kb_cols(df, COLS_CAST[k]) for k, df in tablas.items())
display(Markdown(f"**Reducción de memoria por downcasting (columnas casteadas):** "
                 f"de {antes:.1f} a **{ahora:.1f} KB** (**{(antes-ahora)/antes*100:.1f}%** de ahorro)."))

**Reducción de memoria por downcasting (columnas casteadas):** de 3560.0 a **222.1 KB** (**93.8%** de ahorro).

## **2.10. Gestion para ingesta en MySQL**

### **2.10.1. Exportar tablas para ingesta en MySQL y Notebook 2**

In [350]:
# Directorio actual como raíz del proyecto
ruta_raiz = Path('.')

# Carpetas según nivel de normalización
carpeta_1FN   = ruta_raiz / 'datos_1FN'
carpeta_mysql = ruta_raiz / 'datos_3FN'
carpeta_1FN.mkdir(exist_ok=True)
carpeta_mysql.mkdir(exist_ok=True)
print("✅ Carpetas datos_1FN/ y datos_3FN/ creadas en disco.")

# Tablas a exportar
lista_tablas = [
    df_tabla_info_limpio, df_tabla_tipo_limpio, df_tabla_fake_limpio,
    df_tabla_tiempo_limpio, df_tabla_contenido_limpio, df_tabla_metadatos_limpio,
    df_tabla_pais_idioma_limpio, df_tabla_compartir_tiempo_limpio
]

# Pre-casting: pasa las columnas object a texto puro para evitar errores de PyArrow al escribir Parquet
for df_actual in lista_tablas:
    for col in df_actual.select_dtypes(include=['object']).columns:
        df_actual[col] = df_actual[col].astype(str)
print("✅ Pre-casting completado.")

# Exportación 1FN (Parquet, para el EDA / Notebook 3)
df_tabla_info_limpio.to_parquet(carpeta_1FN / 'info_1FN.parquet', index=False)
df_tabla_tipo_limpio.to_parquet(carpeta_1FN / 'tipo_1FN.parquet', index=False)
df_tabla_fake_limpio.to_parquet(carpeta_1FN / 'fake_1FN.parquet', index=False)
df_tabla_tiempo_limpio.to_parquet(carpeta_1FN / 'tiempo_1FN.parquet', index=False)
df_tabla_contenido_limpio.to_parquet(carpeta_1FN / 'contenido_1FN.parquet', index=False)
df_tabla_metadatos_limpio.to_parquet(carpeta_1FN / 'metadatos_1FN.parquet', index=False)
df_tabla_pais_idioma_limpio.to_parquet(carpeta_1FN / 'pais_idioma_1FN.parquet', index=False)
df_tabla_compartir_tiempo_limpio.to_parquet(carpeta_1FN / 'compartir_tiempo_1FN.parquet', index=False)
print("✅ Datos 1FN exportados (Parquet).")

# Exportación 3FN (CSV, para MySQL / Notebook 2)
df_tabla_info_limpio.to_csv(carpeta_mysql / 'info_3FN.csv', index=False)
df_tabla_tipo_limpio.to_csv(carpeta_mysql / 'tipo_3FN.csv', index=False)
df_tabla_fake_limpio.to_csv(carpeta_mysql / 'fake_3FN.csv', index=False)
df_tabla_tiempo_limpio.to_csv(carpeta_mysql / 'tiempo_3FN.csv', index=False)
df_tabla_contenido_limpio.to_csv(carpeta_mysql / 'contenido_3FN.csv', index=False)
df_tabla_metadatos_limpio.to_csv(carpeta_mysql / 'metadatos_3FN.csv', index=False)
df_tabla_pais_idioma_limpio.to_csv(carpeta_mysql / 'pais_idioma_3FN.csv', index=False)
df_tabla_compartir_tiempo_limpio.to_csv(carpeta_mysql / 'compartir_tiempo_3FN.csv', index=False)
print("✅ Datos 3FN exportados (CSV).")

✅ Carpetas datos_1FN/ y datos_3FN/ creadas en disco.
✅ Pre-casting completado.
✅ Datos 1FN exportados (Parquet).
✅ Datos 3FN exportados (CSV).
